In [3]:
import pandas as pd

In [11]:
# combine 2 fies of product

"""
merge_products.py
-----------------
Combines an OLD product table and a NEW product table into one file.

Rules:
  - Match on product_name (case-insensitive, stripped).  Product IDs are ignored.
  - Duplicates  → keep the NEW table row (new columns format).
  - Unmatched old rows → map old columns → new columns, then:
        • added to the combined output
        • ALSO written to  unmatched_old_products.csv
  - ingredients_list is always preserved.

Usage:
  1. Set OLD_FILE and NEW_FILE to your actual file paths.
  2. Set OUTPUT_FILE and UNMATCHED_FILE as desired.
  3. Run:  python merge_products.py
"""

import pandas as pd
import os

# ── 0. CONFIG ────────────────────────────────────────────────────────────────
OLD_FILE      = "old.csv"       # ← change to your old table path
NEW_FILE      = "new.csv"       # ← change to your new table path
OUTPUT_FILE   = "combined_products.csv"
UNMATCHED_FILE = "unmatched_old_products.csv"


# ── 1. COLUMN MAPPING  (old column → new column) ────────────────────────────
# Only columns that have a logical equivalent in the new schema are mapped.
# Everything else from the old table is dropped (it has no home in the new format).
OLD_TO_NEW = {
    "product_name":       "product_name",
    "brand_name":         "brand_name",
    "product_url":        "product_url",
    "image_url":          "image_url",
    "price":              "price",
    "currency":           "currency",
    "rating":             "rating",
    "review_count":       "number_of_reviews",
    "size":               "size",
    "product_type":       "product_type_text",
    "country":            "country_of_origin",
    "product_description":"description",
    "product_highlights": "product_claims",
    "source_name":        "source",
    "ingredients_list":   "ingredients_list",   # always kept
    # skin type boolean columns → good_for_skin_types (assembled below)
    # skin_type_combination, skin_type_dry, skin_type_normal,
    # skin_type_oily, skin_type_sensitive  handled in build_good_for_skin_types()
}

# Final column order matches the new table + ingredients_list appended
NEW_COLUMNS = [
    "product_id", "product_url", "product_name", "brand_name",
    "category", "sub_category", "image_url", "description",
    "price", "currency", "size", "product_type_text", "key_actives_text",
    "ingredients_count", "rating", "number_of_reviews",
    "good_for_skin_types", "bad_for_skin_types",
    "oily_skin_score", "dry_skin_score", "sensitive_skin_score",
    "combination_skin_score", "normal_skin_score", "acne_prone_score",
    "skin_type_notes", "pregnancy_safe", "fungal_acne_safe",
    "comedogenic_rating", "irritation_rating", "safety_notes",
    "product_claims", "benefits", "concerns", "free_from",
    "vegan", "cruelty_free", "reef_safe", "fragrance_free",
    "alcohol_free", "paraben_free", "sulfate_free", "silicone_free",
    "oil_free", "country_of_origin", "source", "ingredients_list",
]


# ── 2. HELPERS ───────────────────────────────────────────────────────────────
def read_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(path, dtype=str)
    return pd.read_csv(path, dtype=str)


def normalise(name: str) -> str:
    """Lowercase + strip for matching."""
    return str(name).strip().lower() if pd.notna(name) else ""


def build_good_for_skin_types(row: pd.Series) -> str:
    """
    Assembles a comma-separated string from the old boolean skin-type columns.
    e.g.  skin_type_dry=1, skin_type_oily=1  →  "dry, oily"
    """
    mapping = {
        "skin_type_combination": "combination",
        "skin_type_dry":         "dry",
        "skin_type_normal":      "normal",
        "skin_type_oily":        "oily",
        "skin_type_sensitive":   "sensitive",
    }
    types = [
        label for col, label in mapping.items()
        if col in row.index and str(row.get(col, "")).strip() in ("1", "1.0", "True", "true", "yes")
    ]
    return ", ".join(types) if types else None


def map_old_row_to_new(row: pd.Series) -> dict:
    """Convert one old-table row to a dict keyed by new column names."""
    new_row = {col: None for col in NEW_COLUMNS}

    for old_col, new_col in OLD_TO_NEW.items():
        if old_col in row.index:
            new_row[new_col] = row.get(old_col)

    # Assemble good_for_skin_types from boolean columns
    new_row["good_for_skin_types"] = build_good_for_skin_types(row)

    # Count ingredients if list is present
    if new_row["ingredients_list"] and pd.notna(new_row["ingredients_list"]):
        new_row["ingredients_count"] = len(
            str(new_row["ingredients_list"]).split(",")
        )

    return new_row


# ── 3. LOAD ──────────────────────────────────────────────────────────────────
print(f"Reading old table : {OLD_FILE}")
old_df = read_file(OLD_FILE)

print(f"Reading new table : {NEW_FILE}")
new_df = read_file(NEW_FILE)

# Ensure ingredients_list column exists in new table (add if missing)
if "ingredients_list" not in new_df.columns:
    new_df["ingredients_list"] = None

# Ensure all NEW_COLUMNS exist in new_df (fill missing with None)
for col in NEW_COLUMNS:
    if col not in new_df.columns:
        new_df[col] = None

new_df = new_df[NEW_COLUMNS].copy()


# ── 4. MATCH ─────────────────────────────────────────────────────────────────
new_df["_key"] = new_df["product_name"].apply(normalise)
old_df["_key"] = old_df["product_name"].apply(normalise)

matched_keys = set(new_df["_key"])

old_unmatched = old_df[~old_df["_key"].isin(matched_keys)].copy()
old_matched   = old_df[ old_df["_key"].isin(matched_keys)].copy()

print(f"\nNew table rows          : {len(new_df)}")
print(f"Old rows matched (dup)  : {len(old_matched)}  → using new table data")
print(f"Old rows unmatched      : {len(old_unmatched)}  → added to combined + unmatched file")


# ── 5. MAP UNMATCHED OLD ROWS ─────────────────────────────────────────────────
mapped_rows = [map_old_row_to_new(row) for _, row in old_unmatched.iterrows()]
unmatched_new_format = pd.DataFrame(mapped_rows, columns=NEW_COLUMNS)


# ── 6. COMBINE ────────────────────────────────────────────────────────────────
new_df_clean = new_df.drop(columns=["_key"])
combined = pd.concat([new_df_clean, unmatched_new_format], ignore_index=True)


# ── 7. SAVE ───────────────────────────────────────────────────────────────────
combined.to_csv(OUTPUT_FILE, index=False)
print(f"\n✅  Combined file saved  : {OUTPUT_FILE}  ({len(combined)} rows)")

unmatched_new_format.to_csv(UNMATCHED_FILE, index=False)
print(f"✅  Unmatched file saved : {UNMATCHED_FILE}  ({len(unmatched_new_format)} rows)")

Reading old table : old.csv
Reading new table : new.csv

New table rows          : 4801
Old rows matched (dup)  : 2433  → using new table data
Old rows unmatched      : 21400  → added to combined + unmatched file

✅  Combined file saved  : combined_products.csv  (26201 rows)
✅  Unmatched file saved : unmatched_old_products.csv  (21400 rows)


In [12]:
combine=pd.read_csv('combined_products.csv')

C:\Users\DELL\AppData\Local\Temp\ipykernel_8236\1025031093.py:1: DtypeWarning: Columns (0: product_id, 1: category, 2: sub_category, 3: key_actives_text, 4: bad_for_skin_types, 5: skin_type_notes, 6: safety_notes, 7: benefits, 8: concerns, 9: free_from) have mixed types. Specify dtype option on import or set low_memory=False.
  combine=pd.read_csv('combined_products.csv')


In [13]:
c = combine.sort_values(
    by="product_name",
    key=lambda col: col.str.lower(),
    na_position='last'
)

In [14]:
c.to_csv("combine_sorted_products.csv", index=False)

In [15]:
c.columns

Index(['product_id', 'product_url', 'product_name', 'brand_name', 'category',
       'sub_category', 'image_url', 'description', 'price', 'currency', 'size',
       'product_type_text', 'key_actives_text', 'ingredients_count', 'rating',
       'number_of_reviews', 'good_for_skin_types', 'bad_for_skin_types',
       'oily_skin_score', 'dry_skin_score', 'sensitive_skin_score',
       'combination_skin_score', 'normal_skin_score', 'acne_prone_score',
       'skin_type_notes', 'pregnancy_safe', 'fungal_acne_safe',
       'comedogenic_rating', 'irritation_rating', 'safety_notes',
       'product_claims', 'benefits', 'concerns', 'free_from', 'vegan',
       'cruelty_free', 'reef_safe', 'fragrance_free', 'alcohol_free',
       'paraben_free', 'sulfate_free', 'silicone_free', 'oil_free',
       'country_of_origin', 'source', 'ingredients_list'],
      dtype='str')

In [16]:
"""
Product Table Deduplication Script
====================================
Deduplicates a combined product CSV by product_name using:
  1. Heavy normalization (unicode, stopwords, size units, token sort)
  2. Exact match removal on normalized names
  3. Fuzzy matching (blocked by brand) with false-positive protection
     - Protects against removing products with different SPF/%, concentrations,
       day/night, eye, mini, skin type variants

Outputs:
  - combined_deduped.csv       → clean final table
  - duplicates_removed.csv     → all removed rows (exact + fuzzy)
  - uncertain_matches.csv      → pairs needing manual review
"""

import pandas as pd
import re
import unicodedata
from difflib import SequenceMatcher

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE             = "combine_sorted_products.csv"
OUTPUT_DEDUPED         = "combined_deduped.csv"
OUTPUT_REMOVED         = "duplicates_removed.csv"
OUTPUT_UNCERTAIN       = "uncertain_matches.csv"

FUZZY_STRONG_THRESHOLD    = 90   # >= this + passes variant check → duplicate
FUZZY_UNCERTAIN_THRESHOLD = 80   # >= this but < strong → uncertain (manual review)

# ── STOPWORDS (non-differentiating filler words) ──────────────────────────────
STOPWORDS = {
    'the', 'with', 'for', 'and', 'by', 'of', 'a', 'an',
    'in', 'to', 'new', 'my', 'your'
}

# ── Size/volume units to strip (NOT SPF or % numbers) ─────────────────────────
SIZE_UNIT_PATTERN = re.compile(
    r'\b\d+(\.\d+)?\s*(ml|l\b|oz|fl\.?\s*oz|kg|lb|lbs|mm|cm|pack|ct|count'
    r'|pcs?|pieces?|tablets?|capsules?|sachets?|units?)\b',
    re.IGNORECASE
)

# ── Variant keywords: if they differ between two names → NOT a duplicate ───────
VARIANT_KEYWORDS = {
    'day', 'night', 'eye', 'mini', 'travel', 'lite', 'light',
    'original', 'extra', 'plus', 'advanced', 'intensive', 'rich',
    'dry', 'oily', 'sensitive', 'normal', 'combination',
    'tinted', 'untinted', 'sheer', 'matte', 'glow', 'radiant',
}


# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────

def normalize(name: str) -> str:
    """
    Full normalization pipeline:
      - Strip leading/trailing whitespace and newlines
      - Unicode normalization (removes accents: é→e, ü→u, ™→removed)
      - Lowercase
      - Remove size/volume units only (keep SPF numbers, % concentrations)
      - Remove punctuation (keep % sign)
      - Remove stopwords
      - Token sort (makes word order irrelevant)
      - Collapse extra spaces
    """
    if pd.isna(name):
        return ''
    name = str(name).strip().lstrip('\n')

    # Unicode normalization — remove combining characters (accents etc.)
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))

    name = name.lower()

    # Remove size/volume units (ml, oz, kg, etc.) but keep numbers like SPF 50
    name = SIZE_UNIT_PATTERN.sub('', name)

    # Remove punctuation except % (important for concentrations like 2%)
    name = re.sub(r'[^\w\s%]', ' ', name)
    name = name.replace('_', ' ')

    # Remove stopwords
    words = [w for w in name.split() if w not in STOPWORDS]

    # Token sort: makes "Vitamin C Serum Face" == "Face Serum Vitamin C"
    words = sorted(words)

    return re.sub(r'\s+', ' ', ' '.join(words)).strip()


def extract_numbers(name: str) -> set:
    """
    Extract all numeric values from name (SPF values, % concentrations, levels).
    Used to detect products that differ only by number (SPF 30 vs SPF 50).
    """
    return set(re.findall(r'\b\d+(?:\.\d+)?(?:\s*%)?', name.lower()))


def extract_variants(name: str) -> set:
    """
    Extract variant keywords present in name.
    Used to detect products that differ by type (day vs night, eye vs face).
    """
    tokens = set(re.sub(r'[^\w\s]', ' ', name.lower()).split())
    return tokens & VARIANT_KEYWORDS


def is_true_duplicate(name1: str, name2: str) -> bool:
    """
    After fuzzy similarity passes the threshold, apply additional checks
    to reject false positives:
      - If both names contain numbers and they differ → different products
        (e.g. SPF 30 vs SPF 50, Retinol 0.25 vs Retinol 0.5)
      - If variant keywords differ → different products
        (e.g. Day Cream vs Night Cream, Eye Cream vs Face Cream)
    """
    nums1 = extract_numbers(name1)
    nums2 = extract_numbers(name2)
    if nums1 and nums2 and nums1 != nums2:
        return False

    var1 = extract_variants(name1)
    var2 = extract_variants(name2)
    if var1 != var2:
        return False

    return True


def fuzzy_similarity(a: str, b: str) -> float:
    """Compute similarity ratio (0-100) between two strings."""
    return SequenceMatcher(None, a, b).ratio() * 100


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    # ── Load data ──────────────────────────────────────────────────────────────
    print(f"Loading {INPUT_FILE} ...")
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")

    # Drop rows with no product name
    null_names = df['product_name'].isna().sum()
    print(f"length null product {null_names}")
    df = df[df['product_name'].notna()].copy()
    print(f"  Dropped {null_names} rows with null product_name")

    # Clean product_name (strip spaces and leading newlines)
    df['product_name'] = df['product_name'].str.strip().str.lstrip('\n')
    df = df.reset_index(drop=True)

    # ── Step 1: Normalize ──────────────────────────────────────────────────────
    print("\nNormalizing product names ...")
    df['norm_name'] = df['product_name'].apply(normalize)

    # ── Step 2: Exact dedup on normalized name ─────────────────────────────────
    exact_dup_mask = df.duplicated(subset='norm_name', keep='first')
    exact_dups_df  = df[exact_dup_mask].copy()
    df_clean       = df[~exact_dup_mask].copy().reset_index(drop=True)
    print(f"  Exact duplicates removed: {exact_dup_mask.sum():,}")
    print(f"  Remaining for fuzzy check: {len(df_clean):,}")

    # ── Step 3: Fuzzy matching blocked by brand ────────────────────────────────
    print("\nRunning fuzzy matching (blocked by brand) ...")
    df_clean['brand_norm'] = (
        df_clean['brand_name'].fillna('unknown').str.lower().str.strip()
    )

    fuzzy_strong    = []   # confirmed duplicates
    fuzzy_uncertain = []   # needs manual review
    to_drop         = set()

    total_brands = df_clean['brand_norm'].nunique()
    for i, (brand, grp) in enumerate(df_clean.groupby('brand_norm')):
        if i % 500 == 0:
            print(f"  Processing brand {i+1}/{total_brands} ...", end='\r')

        idxs  = grp.index.tolist()
        norms = grp['norm_name'].tolist()
        names = grp['product_name'].tolist()

        for a_i in range(len(idxs)):
            if idxs[a_i] in to_drop:
                continue
            for b_i in range(a_i + 1, len(idxs)):
                if idxs[b_i] in to_drop:
                    continue

                norm_a, norm_b = norms[a_i], norms[b_i]
                if not norm_a or not norm_b:
                    continue

                score = fuzzy_similarity(norm_a, norm_b)

                if score >= FUZZY_STRONG_THRESHOLD:
                    if is_true_duplicate(names[a_i], names[b_i]):
                        # Confirmed duplicate — drop the second one
                        fuzzy_strong.append({
                            'kept_idx'    : idxs[a_i],
                            'dropped_idx' : idxs[b_i],
                            'kept_name'   : names[a_i],
                            'dropped_name': names[b_i],
                            'brand'       : brand,
                            'similarity'  : round(score, 1),
                            'reason'      : 'fuzzy_strong_duplicate'
                        })
                        to_drop.add(idxs[b_i])
                    else:
                        # High similarity but different variant/number — flag for review
                        fuzzy_uncertain.append({
                            'name_1'    : names[a_i],
                            'name_2'    : names[b_i],
                            'brand'     : brand,
                            'similarity': round(score, 1),
                            'reason'    : 'different_numbers_or_variants'
                        })

                elif score >= FUZZY_UNCERTAIN_THRESHOLD:
                    # Borderline — flag for manual review
                    fuzzy_uncertain.append({
                        'name_1'    : names[a_i],
                        'name_2'    : names[b_i],
                        'brand'     : brand,
                        'similarity': round(score, 1),
                        'reason'    : 'borderline_similarity'
                    })

    print(f"\n  Fuzzy strong duplicates (≥{FUZZY_STRONG_THRESHOLD}%): {len(fuzzy_strong):,}")
    print(f"  Moved to uncertain (≥{FUZZY_STRONG_THRESHOLD}% but diff variant/number): "
          f"{sum(1 for u in fuzzy_uncertain if u['reason'] == 'different_numbers_or_variants'):,}")
    print(f"  Borderline uncertain ({FUZZY_UNCERTAIN_THRESHOLD}-{FUZZY_STRONG_THRESHOLD-1}%): "
          f"{sum(1 for u in fuzzy_uncertain if u['reason'] == 'borderline_similarity'):,}")

    # ── Step 4: Build output dataframes ───────────────────────────────────────
    drop_cols = ['norm_name', 'brand_norm']

    df_deduped       = df_clean[~df_clean.index.isin(to_drop)].drop(columns=drop_cols)
    df_fuzzy_removed = df_clean[df_clean.index.isin(to_drop)].drop(columns=drop_cols)
    all_removed      = pd.concat(
        [exact_dups_df.drop(columns=['norm_name']), df_fuzzy_removed],
        ignore_index=True
    )
    df_uncertain = pd.DataFrame(fuzzy_uncertain)[
        ['name_1', 'name_2', 'brand', 'similarity', 'reason']
    ]

    # ── Step 5: Save outputs ───────────────────────────────────────────────────
    df_deduped.to_csv(OUTPUT_DEDUPED,   index=False)
    all_removed.to_csv(OUTPUT_REMOVED,  index=False)
    df_uncertain.to_csv(OUTPUT_UNCERTAIN, index=False)

    print(f"\n{'='*55}")
    print(f"  ✅ combined_deduped.csv      → {len(df_deduped):,} rows")
    print(f"  🗑️  duplicates_removed.csv   → {len(all_removed):,} rows")
    print(f"  ⚠️  uncertain_matches.csv    → {len(df_uncertain):,} pairs")
    print(f"{'='*55}")

    print("\nSample confirmed fuzzy duplicates removed:")
    for r in fuzzy_strong[:5]:
        print(f"  [{r['similarity']}%] KEPT: {r['kept_name']!r}")
        print(f"         DROP: {r['dropped_name']!r}")


if __name__ == "__main__":
    main()

Loading combine_sorted_products.csv ...


  Loaded 26,201 rows, 46 columns
length null product 1
  Dropped 1 rows with null product_name

Normalizing product names ...
  Exact duplicates removed: 890
  Remaining for fuzzy check: 25,310

Running fuzzy matching (blocked by brand) ...
  Processing brand 2501/2557 ...
  Fuzzy strong duplicates (≥90%): 410
  Moved to uncertain (≥90% but diff variant/number): 403
  Borderline uncertain (80-89%): 2,933

  ✅ combined_deduped.csv      → 24,900 rows
  🗑️  duplicates_removed.csv   → 1,300 rows
  ⚠️  uncertain_matches.csv    → 3,336 pairs

Sample confirmed fuzzy duplicates removed:
  [93.3%] KEPT: 'Madecassoside Ampoule'
         DROP: 'Madecassoside Ampoule 2x'
  [92.7%] KEPT: 'Madecassoside Cream'
         DROP: 'Madecassoside Cream 2x'
  [90.7%] KEPT: 'Heartleaf Sun Essence Calming Drop'
         DROP: 'Heartleaf Sun Essence Calming Drop SPF 50+'
  [90.0%] KEPT: 'Aesop Geranium Leaf Body Cleanser 500ml'
         DROP: 'Geranium Leaf Body Cleanser'
  [98.2%] KEPT: 'Ato Barrier 365 Hydro

In [17]:
c_d=pd.read_csv('combined_deduped.csv')  ###herereeeeeeeeeeeeee

In [18]:
c1= c_d.sort_values(
    by="product_name",
    key=lambda col: col.str.lower(),
    na_position='last'
)



In [19]:
c1.to_csv("final_combined_after_dedpued.csv", index=False)

In [20]:
len(c1)

24900

In [21]:
# Generate unique id for every product depend on brand and product name 
"""
Product ID Generator
=====================
Generates a stable, unique hash-based ID for every product row
using product_name + brand_name as the input.

- Same product always gets the same ID (deterministic)
- Safe to use as a stable foreign key in a database
- 16-character hex ID from MD5 hash (18+ quadrillion combinations)

Input:  final_combined_after_dedpued.csv  (or any product CSV)
Output: final_combinedlast.csv  (product_id column replaced, moved to first column)
"""

import pandas as pd
import hashlib

# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE  = "final_combined_after_dedpued.csv"
OUTPUT_FILE = "final_combinedlast.csv"   # overwrite in place; change if needed
ID_LENGTH   = 16                       # characters to take from MD5 hash


# ── ID GENERATION ─────────────────────────────────────────────────────────────

def generate_id(row: pd.Series) -> str:
    """
    Generate a stable 16-character hex ID from product_name + brand_name.

    Formula:
        MD5( lowercase(brand_name) + "|" + lowercase(product_name) )
        → first 16 hex characters

    The "|" separator prevents brand "AB" + name "CD" from colliding
    with brand "A" + name "BCD".

    If brand_name is missing, only product_name is used.
    """
    name  = str(row['product_name']).strip().lower()
    brand = str(row['brand_name']).strip().lower() if pd.notna(row['brand_name']) else ''
    raw   = f"{brand}|{name}"
    return hashlib.md5(raw.encode('utf-8')).hexdigest()[:ID_LENGTH]


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    print(f"Loading {INPUT_FILE} ...")
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")

    # Generate IDs
    print("\nGenerating hash-based product IDs ...")
    df['product_id'] = df.apply(generate_id, axis=1)

    # Verify uniqueness
    total      = len(df)
    unique_ids = df['product_id'].nunique()
    collisions = total - unique_ids

    print(f"  Total rows:    {total:,}")
    print(f"  Unique IDs:    {unique_ids:,}")
    print(f"  ID collisions: {collisions}")

    if collisions > 0:
        print("\n  ⚠️  Collisions detected — likely duplicate product_name + brand_name pairs.")
        print("  Consider deduplicating first or increasing ID_LENGTH.")

    # Move product_id to first column
    cols = ['product_id'] + [c for c in df.columns if c != 'product_id']
    df   = df[cols]

    # Save
    df.to_csv(OUTPUT_FILE, index=False)
    print(f"\n✅ Saved {OUTPUT_FILE} with new product_id column.")

    # Preview
    print("\nSample output:")
    print(df[['product_id', 'product_name', 'brand_name']].head(8).to_string(index=False))


if __name__ == "__main__":
    main()

Loading final_combined_after_dedpued.csv ...
  Loaded 24,900 rows, 46 columns

Generating hash-based product IDs ...
  Total rows:    24,900
  Unique IDs:    24,900
  ID collisions: 0

✅ Saved final_combinedlast.csv with new product_id column.

Sample output:
      product_id                               product_name           brand_name
fd4663ee5bdc317f                                    "B" Oil         The Ordinary
224e919a4ef5c363                                   "Buffet"         The Ordinary
34e104e46244fe98              "Buffet" + Copper Peptides 1%         The Ordinary
eff06583ca211a3c    "Cocoa Kisses" Whipped Sugar Body Scrub Soap Cute California
6d0dfddb657c41a0      "Cotton Candy Clouds" After Shave Oil Soap Cute California
8388cd1fe80cf2d9 "Cotton Candy Clouds" Whipped Shave Butter Soap Cute California
a74836a8bd38a0dd            "Day In Malibu" After Shave Oil Soap Cute California
c9ce61a3d74d2b77       "Day In Malibu" Whipped Shave Butter Soap Cute California


In [22]:
"""
Script 1 — Build Unified Ingredients Table (16-char IDs)
==========================================================
Builds the final ingredients table with ALL IDs in the same
16-char MD5 format, regardless of source.

Steps:
  1. Load existing ingredients table (skincarisma — 32-char IDs)
  2. Regenerate ALL existing IDs to 16-char format (MD5 of ingredient_name)
  3. Extract ingredients from product table's ingredients_list column
  4. Filter garbage/noise strings
  5. Match extracted ingredients against existing table
  6. Append only new (unmatched) ingredients with 16-char IDs
  7. Verify no ID collisions

Inputs:
  - PRODUCTS_FILE    → product table (with ingredients_list column)
  - INGREDIENTS_FILE → existing ingredients table (skincarisma)

Outputs:
  - ingredients_unified.csv      → full ingredients table, ALL IDs 16-char
  - ingredients_id_map.csv       → old_id → new_id mapping (needed for Script 2)
  - ingredients_matched.csv      → extracted names that matched existing rows
  - ingredients_filtered_out.csv → noise/garbage filtered before matching
"""

import pandas as pd
import ast
import re
import hashlib
import unicodedata

# ── CONFIG ────────────────────────────────────────────────────────────────────
PRODUCTS_FILE    = "final_combinedlast.csv"
INGREDIENTS_FILE = "ingredients.csv"

OUTPUT_INGREDIENTS = "ingredients_unified.csv"
OUTPUT_ID_MAP      = "ingredients_id_map.csv"       # ⚠️ needed for Script 2
OUTPUT_MATCHED     = "ingredients_matched.csv"
OUTPUT_FILTERED    = "ingredients_filtered_out.csv"

ID_LENGTH = 16


# ── GARBAGE FILTER ────────────────────────────────────────────────────────────
GARBAGE_PATTERNS = [
    r'^#',                   # Excel errors: #NAME?, #REF!
    r'ingredients\s*:',      # Label strings: "Ingredients: Water..."
    r'étape\s*\d',           # French step labels
    r'step\s*\d',            # English step labels
    r'^\(\*',                # (* - exfoliant) annotation labels
    r'^-\s*exfoliant',
    r'^\d+$',                # Pure numbers: "1", "2"
    r'^[^a-zA-Z\(\*]',      # Starts with non-letter (except parentheses)
]
GARBAGE_RE = re.compile('|'.join(GARBAGE_PATTERNS), re.IGNORECASE)


# ── HELPERS ───────────────────────────────────────────────────────────────────

def make_id(name: str) -> str:
    """Stable 16-char hex ID — MD5 of lowercase ingredient name."""
    return hashlib.md5(name.strip().lower().encode('utf-8')).hexdigest()[:ID_LENGTH]


def normalize_ing(name: str) -> str:
    """Normalize ingredient name for matching (unicode + lowercase + strip)."""
    name = unicodedata.normalize('NFKD', str(name))
    name = ''.join(c for c in name if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', name.lower().strip())


def is_valid_ingredient(name: str) -> bool:
    """Return True if string looks like a real ingredient name."""
    name = name.strip()
    if len(name) < 3:             return False
    if GARBAGE_RE.search(name):   return False
    if name.count(' ') > 15:      return False
    return True


def parse_ingredients(val) -> list:
    """Parse ingredients_list cell → list of ingredient name strings."""
    if pd.isna(val):
        return []
    s = str(val).strip()
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return [str(i).strip() for i in parsed if str(i).strip()]
    except Exception:
        pass
    return [i.strip().strip("'\"") for i in re.split(r',(?=\s)', s) if i.strip()]


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():

    # ── Load ──────────────────────────────────────────────────────────────────
    print(f"Loading ingredients from : {INGREDIENTS_FILE}")
    ing = pd.read_csv(INGREDIENTS_FILE, low_memory=False)
    print(f"  {len(ing):,} existing ingredients loaded")

    print(f"Loading products from    : {PRODUCTS_FILE}")
    prod = pd.read_csv(PRODUCTS_FILE, low_memory=False)
    print(f"  {len(prod):,} products loaded")

    # ── Step 1: Unify existing ingredient IDs to 16-char ──────────────────────
    print("\nStep 1: Regenerating all existing ingredient IDs to 16-char ...")

    # Show before
    ing['_old_id_len'] = ing['ingredient_id'].astype(str).apply(len)
    print("  ID length distribution BEFORE:")
    for length, count in ing.groupby('_old_id_len').size().items():
        print(f"    {length}-char : {count:,} rows")

    # Save old → new mapping (Script 2 will use this to update the bridge)
    ing['old_ingredient_id'] = ing['ingredient_id'].astype(str)
    ing['ingredient_id']     = ing['ingredient_name'].apply(make_id)

    id_map = ing[['old_ingredient_id', 'ingredient_id', 'ingredient_name']].copy()
    id_map.columns = ['old_id', 'new_id', 'ingredient_name']

    # Show after
    ing['_new_id_len'] = ing['ingredient_id'].astype(str).apply(len)
    print("  ID length distribution AFTER:")
    for length, count in ing.groupby('_new_id_len').size().items():
        flag = '✅' if length == ID_LENGTH else '❌'
        print(f"    {flag} {length}-char : {count:,} rows")

    # Check collisions
    collisions = len(ing) - ing['ingredient_id'].nunique()
    print(f"  {'✅' if collisions == 0 else '❌'} ID collisions: {collisions}")
    if collisions > 0:
        print("  ⚠️  Duplicate ingredient names found — these will share an ID:")
        dupes = ing[ing.duplicated(subset='ingredient_id', keep=False)]
        print(dupes[['ingredient_id', 'ingredient_name', 'source']].head(10).to_string())

    # Clean temp columns
    ing = ing.drop(columns=['_old_id_len', '_new_id_len', 'old_ingredient_id'])

    # ── Step 2: Extract ingredients from product table ─────────────────────────
    print("\nStep 2: Extracting ingredients from products ...")
    all_raw = set()
    for val in prod['ingredients_list'].dropna():
        for name in parse_ingredients(val):
            if name:
                all_raw.add(name.strip())

    valid_extracted   = {n for n in all_raw if is_valid_ingredient(n)}
    invalid_extracted = all_raw - valid_extracted

    print(f"  Total unique raw values   : {len(all_raw):,}")
    print(f"  Valid ingredient names    : {len(valid_extracted):,}")
    print(f"  Filtered as garbage       : {len(invalid_extracted):,}")

    # ── Step 3: Match against unified existing table ───────────────────────────
    print("\nStep 3: Matching extracted ingredients against existing table ...")
    ing['_norm']   = ing['ingredient_name'].apply(normalize_ing)
    existing_norms = set(ing['_norm'])

    matched   = []
    new_names = []

    for raw_name in sorted(valid_extracted):
        if normalize_ing(raw_name) in existing_norms:
            matched.append(raw_name)
        else:
            new_names.append(raw_name)

    print(f"  Matched (already in table) : {len(matched):,}")
    print(f"  New (to be added)          : {len(new_names):,}")

    # ── Step 4: Build new ingredient rows (all 16-char IDs) ───────────────────
    new_rows = pd.DataFrame({
        'ingredient_id'  : [make_id(n) for n in new_names],
        'ingredient_name': new_names,
        'description'    : None,
        'evidence_level' : None,
        'science_tags'   : None,
        'science_details': None,
        'callout_type'   : None,
        'callout_text'   : None,
        'warning_type'   : None,
        'warning_text'   : None,
        'source'         : 'extracted_from_products'
    })

    # ── Step 5: Build final ingredients table ─────────────────────────────────
    ing_clean   = ing.drop(columns=['_norm'])
    updated_ing = pd.concat([ing_clean, new_rows], ignore_index=True)

    # Final ID length verification
    updated_ing['_id_len'] = updated_ing['ingredient_id'].astype(str).apply(len)
    print("\n  Final ID length distribution (all must be 16):")
    for length, count in updated_ing.groupby('_id_len').size().items():
        flag = '✅' if length == ID_LENGTH else '❌'
        print(f"    {flag} {length}-char : {count:,} rows")
    updated_ing = updated_ing.drop(columns=['_id_len'])

    final_collisions = len(updated_ing) - updated_ing['ingredient_id'].nunique()

    # ── Save outputs ──────────────────────────────────────────────────────────
    updated_ing.to_csv(OUTPUT_INGREDIENTS, index=False)
    id_map.to_csv(OUTPUT_ID_MAP, index=False)
    pd.DataFrame({'extracted_name': matched, 'status': 'matched_existing'}).to_csv(OUTPUT_MATCHED, index=False)
    pd.DataFrame({'ingredient_name': sorted(invalid_extracted), 'reason': 'filtered_garbage'}).to_csv(OUTPUT_FILTERED, index=False)

    print(f"\n{'=' * 55}")
    print(f"  Original rows (ID unified) : {len(ing_clean):,}")
    print(f"  New rows added             : {len(new_rows):,}")
    print(f"  ✅ Total rows              : {len(updated_ing):,}")
    print(f"  {'✅' if final_collisions == 0 else '❌'} ID collisions        : {final_collisions}")
    print(f"{'=' * 55}")
    print(f"\nSaved:")
    print(f"  → {OUTPUT_INGREDIENTS}   (full ingredients table — all IDs 16-char)")
    print(f"  → {OUTPUT_ID_MAP}        (old→new ID map — ⚠️  needed for Script 2)")
    print(f"  → {OUTPUT_MATCHED}")
    print(f"  → {OUTPUT_FILTERED}")


if __name__ == "__main__":
    main()

Loading ingredients from : ingredients.csv
  17,569 existing ingredients loaded
Loading products from    : final_combinedlast.csv
  24,900 products loaded

Step 1: Regenerating all existing ingredient IDs to 16-char ...
  ID length distribution BEFORE:
    32-char : 17,569 rows
  ID length distribution AFTER:
    ✅ 16-char : 17,569 rows
  ✅ ID collisions: 0

Step 2: Extracting ingredients from products ...
  Total unique raw values   : 22,161
  Valid ingredient names    : 21,457
  Filtered as garbage       : 704

Step 3: Matching extracted ingredients against existing table ...
  Matched (already in table) : 6,662
  New (to be added)          : 14,795

  Final ID length distribution (all must be 16):
    ✅ 16-char : 32,364 rows

  Original rows (ID unified) : 17,569
  New rows added             : 14,795
  ✅ Total rows              : 32,364
  ❌ ID collisions        : 442

Saved:
  → ingredients_unified.csv   (full ingredients table — all IDs 16-char)
  → ingredients_id_map.csv        (o

In [23]:
"""
Fix Ingredient ID Collisions — Keep Skincarisma, Drop Extracted Duplicates
===========================================================================
Fixes the 442 ID collisions in ingredients_unified.csv produced by Script 1.

When two rows share the same ingredient_id (collision):
  - Keep the skincarisma row (has more data: descriptions, evidence, etc.)
  - Drop the extracted_from_products row
  - If both rows are extracted_from_products, keep the first alphabetically

Also updates ingredients_id_map.csv so Script 2 (bridge update) stays correct.

Inputs:
  - INGREDIENTS_FILE → ingredients_unified.csv (output of Script 1, has collisions)
  - ID_MAP_FILE      → ingredients_id_map.csv  (output of Script 1)

Outputs:
  - ingredients_final.csv         → clean ingredients table, 0 collisions, all 16-char
  - ingredients_id_map_final.csv  → updated id map (for Script 2 bridge update)
  - ingredients_collisions.csv    → the 442 dropped rows (for your reference)
"""

import pandas as pd

# ── CONFIG ────────────────────────────────────────────────────────────────────
INGREDIENTS_FILE = "ingredients_unified.csv"
ID_MAP_FILE      = "ingredients_id_map.csv"

OUTPUT_INGREDIENTS = "ingredients_unified.csv"
OUTPUT_ID_MAP      = "ingredients_id_map_final.csv"
OUTPUT_COLLISIONS  = "ingredients_collisions.csv"

# Source priority — lower number = higher priority (kept over others)
SOURCE_PRIORITY = {
    'skincarisma'            : 0,
    'extracted_from_products': 1,
}


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():

    # ── Load ──────────────────────────────────────────────────────────────────
    print(f"Loading : {INGREDIENTS_FILE}")
    ing = pd.read_csv(INGREDIENTS_FILE, low_memory=False)
    print(f"  {len(ing):,} rows loaded")

    id_map = pd.read_csv(ID_MAP_FILE, low_memory=False)
    print(f"  {len(id_map):,} ID map entries loaded")

    # ── Check collisions before fix ───────────────────────────────────────────
    total      = len(ing)
    unique_ids = ing['ingredient_id'].nunique()
    collisions = total - unique_ids

    print(f"\nBEFORE fix:")
    print(f"  Total rows      : {total:,}")
    print(f"  Unique IDs      : {unique_ids:,}")
    print(f"  ❌ Collisions   : {collisions:,}")

    if collisions == 0:
        print("  ✅ No collisions found — nothing to fix.")
        ing.to_csv(OUTPUT_INGREDIENTS, index=False)
        id_map.to_csv(OUTPUT_ID_MAP, index=False)
        return

    # ── Show sample collisions ────────────────────────────────────────────────
    dup_mask = ing.duplicated(subset='ingredient_id', keep=False)
    collision_rows = ing[dup_mask].sort_values(['ingredient_id', 'source'])

    print(f"\n  Sample collisions:")
    shown = set()
    for _, row in collision_rows.iterrows():
        iid = row['ingredient_id']
        if iid not in shown:
            group = collision_rows[collision_rows['ingredient_id'] == iid]
            print(f"    ID: {iid}")
            for _, r in group.iterrows():
                print(f"      [{r['source']}] {r['ingredient_name']!r}")
            shown.add(iid)
            if len(shown) >= 5:
                break

    # ── Fix: assign priority and keep best row per ingredient_id ──────────────
    ing['_priority'] = ing['source'].map(SOURCE_PRIORITY).fillna(99)

    # Sort: lower priority number first (skincarisma=0 before extracted=1)
    # then alphabetically by ingredient_name as tiebreaker
    ing_sorted = ing.sort_values(
        by=['ingredient_id', '_priority', 'ingredient_name'],
        ascending=[True, True, True]
    )

    # Keep first row per ingredient_id (= highest priority = skincarisma)
    ing_clean   = ing_sorted.drop_duplicates(subset='ingredient_id', keep='first')
    ing_dropped = ing_sorted[ing_sorted.duplicated(subset='ingredient_id', keep='first')]

    ing_clean = ing_clean.drop(columns=['_priority'])
    ing_dropped = ing_dropped.drop(columns=['_priority'])

    # ── Verify after fix ──────────────────────────────────────────────────────
    after_collisions = len(ing_clean) - ing_clean['ingredient_id'].nunique()
    all_16_char = (ing_clean['ingredient_id'].astype(str).apply(len) == 16).all()

    print(f"\nAFTER fix:")
    print(f"  Total rows      : {len(ing_clean):,}")
    print(f"  Rows dropped    : {len(ing_dropped):,}  (extracted duplicates)")
    print(f"  {'✅' if after_collisions == 0 else '❌'} Collisions   : {after_collisions}")
    print(f"  {'✅' if all_16_char else '❌'} All IDs 16-char")

    print(f"\n  Dropped breakdown by source:")
    for src, count in ing_dropped['source'].value_counts().items():
        print(f"    {src} : {count:,}")

    # ── Update id_map ─────────────────────────────────────────────────────────
    # For dropped rows, their ingredient_id now points to the kept row's data
    # The mapping stays valid — old_id → new_id is unchanged
    # But we add a note about which dropped names map to which kept name

    kept_id_to_name = dict(zip(ing_clean['ingredient_id'], ing_clean['ingredient_name']))

    id_map['kept_ingredient_name'] = id_map['new_id'].map(kept_id_to_name)
    id_map['was_collision']        = ~(
        id_map['ingredient_name'] == id_map['kept_ingredient_name']
    )

    collision_count_map = id_map['was_collision'].sum()
    print(f"\n  ID map entries flagged as collisions: {collision_count_map:,}")

    # ── Save outputs ──────────────────────────────────────────────────────────
    ing_clean.to_csv(OUTPUT_INGREDIENTS, index=False)
    id_map.to_csv(OUTPUT_ID_MAP, index=False)
    ing_dropped.to_csv(OUTPUT_COLLISIONS, index=False)

    print(f"\n{'=' * 55}")
    print(f"  ✅ {OUTPUT_INGREDIENTS}")
    print(f"     {len(ing_clean):,} rows — 0 collisions — all IDs 16-char")
    print(f"  ✅ {OUTPUT_ID_MAP}")
    print(f"     {len(id_map):,} entries — updated with collision flags")
    print(f"  ℹ️  {OUTPUT_COLLISIONS}")
    print(f"     {len(ing_dropped):,} dropped rows saved for reference")
    print(f"{'=' * 55}")
    # print(f"\n⚠️  Next step: run Script 2 (bridge update) using:")
    # print(f"     INGREDIENTS_FILE = '{OUTPUT_INGREDIENTS}'")
    # print(f"     ID_MAP_FILE      = '{OUTPUT_ID_MAP}'")


if __name__ == "__main__":
    main()

Loading : ingredients_unified.csv
  32,364 rows loaded
  17,569 ID map entries loaded

BEFORE fix:
  Total rows      : 32,364
  Unique IDs      : 31,922
  ❌ Collisions   : 442

  Sample collisions:
    ID: 02be2e911b0e9c53
      [extracted_from_products] 'PEG-10 Laurate'
      [extracted_from_products] 'Peg-10 Laurate'
    ID: 03891c7849c2a70b
      [extracted_from_products] 'Glucose Syrup'
      [extracted_from_products] 'glucose syrup'
    ID: 039d41c6a1acf758
      [extracted_from_products] 'CHONDRUS CRISPUS POWDER (CARRAGEENAN)'
      [extracted_from_products] 'Chondrus Crispus Powder (Carrageenan)'
    ID: 0527b677a99adae8
      [extracted_from_products] 'Citrus Sinensis (Sweet Orange) Oil'
      [extracted_from_products] 'Citrus sinensis (Sweet Orange) Oil'
    ID: 07681f8a32cc2342
      [extracted_from_products] 'Trisodium EthylenediamineDisuccinate'
      [extracted_from_products] 'Trisodium Ethylenediaminedisuccinate'

AFTER fix:
  Total rows      : 31,922
  Rows dropped    : 

In [24]:
ing=pd.read_csv('ingredients_unified.csv')

In [25]:
s_ings= ing.sort_values(
    by="ingredient_name",
    key=lambda col: col.str.lower(),
    na_position='last'
)
s_ings.to_csv("ingredients_updatedlast.csv", index=False)

In [26]:
d=pd.read_csv('ingredients_updatedlast.csv')

In [27]:
len(d)

31922

In [28]:
d.columns

Index(['ingredient_id', 'ingredient_name', 'description', 'evidence_level',
       'science_tags', 'science_details', 'callout_type', 'callout_text',
       'warning_type', 'warning_text', 'source'],
      dtype='str')

In [29]:
d['ingredient_id'].nunique()

31922

In [30]:
bri=pd.read_csv('bridge.csv')

In [31]:
bri.columns

Index(['product_id', 'product_name', 'ingredient_id', 'ingredient_name',
       'ingredient_position', 'ingredient_concentration',
       'raw_ingredient_name', 'evidence_level', 'source'],
      dtype='str')

In [32]:
i=pd.read_csv('ingredients_updatedlast.csv')

In [33]:
i.columns

Index(['ingredient_id', 'ingredient_name', 'description', 'evidence_level',
       'science_tags', 'science_details', 'callout_type', 'callout_text',
       'warning_type', 'warning_text', 'source'],
      dtype='str')

In [34]:
p=pd.read_csv('final_combinedlast.csv')

In [35]:
p.columns

Index(['product_id', 'product_url', 'product_name', 'brand_name', 'category',
       'sub_category', 'image_url', 'description', 'price', 'currency', 'size',
       'product_type_text', 'key_actives_text', 'ingredients_count', 'rating',
       'number_of_reviews', 'good_for_skin_types', 'bad_for_skin_types',
       'oily_skin_score', 'dry_skin_score', 'sensitive_skin_score',
       'combination_skin_score', 'normal_skin_score', 'acne_prone_score',
       'skin_type_notes', 'pregnancy_safe', 'fungal_acne_safe',
       'comedogenic_rating', 'irritation_rating', 'safety_notes',
       'product_claims', 'benefits', 'concerns', 'free_from', 'vegan',
       'cruelty_free', 'reef_safe', 'fragrance_free', 'alcohol_free',
       'paraben_free', 'sulfate_free', 'silicone_free', 'oil_free',
       'country_of_origin', 'source', 'ingredients_list'],
      dtype='str')

In [36]:
# building bridge table 
"""
Build & Validate Skincarisma Bridge Table
==========================================
The existing bridge.csv uses old 32-char IDs for both product_id
and ingredient_id. This script:

  Step 1 — Remap product_id:
            Match bridge rows to product table by product_name
            Replace old 32-char product_id with new 16-char product_id

  Step 2 — Remap ingredient_id:
            Match bridge rows to ingredients table by ingredient_name
            Replace old 32-char ingredient_id with new 16-char ingredient_id

  Step 3 — Drop unmatched rows:
            Rows where product_name not found in product table → dropped
            (ingredient_name match is 100% so no drops needed there)

  Step 4 — Drop skincarisma products with no bridge rows:
            Products that lost all their bridge rows in Step 3 → dropped
            from product table

  Step 5 — FK Integrity Check:
            - bridge.product_id    → skincarisma products  (must be 0 violations)
            - bridge.ingredient_id → skincarisma ingredients (must be 0 violations)
            - every skincarisma product has at least one bridge row
            - no null IDs
            - all IDs are 16-char

Inputs:
  - PRODUCTS_FILE      → final_combinedlast.csv      (16-char product IDs)
  - INGREDIENTS_FILE   → ingredients_updatedlast.csv     (16-char ingredient IDs)
  - BRIDGE_FILE        → bridge.csv                  (32-char IDs — to be remapped)

Outputs:
  - bridge_skinca_final.csv           → clean skincarisma bridge (16-char IDs)
  - products_final.csv                → product table after dropping uncovered products
  - bridge_dropped_rows.csv           → rows dropped (no product match)
  - products_dropped.csv              → skincarisma products dropped (no bridge rows)
"""

import pandas as pd
import unicodedata
import re

# ── CONFIG ────────────────────────────────────────────────────────────────────
PRODUCTS_FILE    = "final_combinedlast.csv"
INGREDIENTS_FILE = "ingredients_updatedlast.csv"
BRIDGE_FILE      = "bridge.csv"

OUTPUT_BRIDGE           = "bridge_skinca_final.csv"
OUTPUT_PRODUCTS         = "final_combinedlast.csv"
OUTPUT_DROPPED_ROWS     = "bridge_dropped_rows.csv"
OUTPUT_DROPPED_PRODUCTS = "products_dropped.csv"


# ── HELPERS ───────────────────────────────────────────────────────────────────

def normalize(name) -> str:
    """Unicode normalization + lowercase + strip whitespace."""
    if pd.isna(name):
        return ''
    name = unicodedata.normalize('NFKD', str(name))
    name = ''.join(c for c in name if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', name.lower().strip())


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():

    # ── Load ──────────────────────────────────────────────────────────────────
    print("Loading tables ...")
    prod   = pd.read_csv(PRODUCTS_FILE,    low_memory=False)
    ing    = pd.read_csv(INGREDIENTS_FILE, low_memory=False)
    bridge = pd.read_csv(BRIDGE_FILE,      low_memory=False)

    print(f"  Products     : {len(prod):,}")
    print(f"  Ingredients  : {len(ing):,}")
    print(f"  Bridge       : {len(bridge):,}")

    # Scope: skincarisma only
    prod_skinca = prod[prod['source'] == 'skincarisma'].copy()
    ing_skinca  = ing[ing['source']   == 'skincarisma'].copy()

    print(f"\n  Skincarisma products    : {len(prod_skinca):,}")
    print(f"  Skincarisma ingredients : {len(ing_skinca):,}")

    # Show bridge ID format before
    print(f"\n  Bridge ID format BEFORE remap:")
    print(f"    product_id lengths    : {sorted(bridge['product_id'].astype(str).apply(len).unique())}-char")
    print(f"    ingredient_id lengths : {sorted(bridge['ingredient_id'].astype(str).apply(len).unique())}-char")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 1 — Remap product_id via product_name
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "─" * 60)
    print("STEP 1: Remapping product_id (32-char → 16-char)")
    print("─" * 60)

    prod_skinca['_norm'] = prod_skinca['product_name'].apply(normalize)
    bridge['_norm_prod'] = bridge['product_name'].apply(normalize)

    name_to_new_prod_id = dict(zip(prod_skinca['_norm'], prod_skinca['product_id']))
    bridge['product_id'] = bridge['_norm_prod'].map(name_to_new_prod_id)

    matched_bridge   = bridge[bridge['product_id'].notna()].copy()
    dropped_bridge   = bridge[bridge['product_id'].isna()].copy()

    print(f"  ✅ Remapped successfully : {len(matched_bridge):,} rows")
    print(f"  ❌ No product match      : {len(dropped_bridge):,} rows  → dropped")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 2 — Remap ingredient_id via ingredient_name
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "─" * 60)
    print("STEP 2: Remapping ingredient_id (32-char → 16-char)")
    print("─" * 60)

    ing_skinca['_norm'] = ing_skinca['ingredient_name'].apply(normalize)
    matched_bridge['_norm_ing'] = matched_bridge['ingredient_name'].apply(normalize)

    name_to_new_ing_id = dict(zip(ing_skinca['_norm'], ing_skinca['ingredient_id']))
    matched_bridge['ingredient_id'] = matched_bridge['_norm_ing'].map(name_to_new_ing_id)

    # Also update ingredient_name to canonical name in ingredients table
    id_to_canonical_name = dict(zip(ing_skinca['ingredient_id'], ing_skinca['ingredient_name']))
    matched_bridge['ingredient_name'] = matched_bridge['ingredient_id'].map(id_to_canonical_name)

    ing_not_found = matched_bridge['ingredient_id'].isna().sum()
    print(f"  ✅ Remapped successfully : {len(matched_bridge) - ing_not_found:,} rows")
    print(f"  ❌ No ingredient match   : {ing_not_found:,} rows")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 3 — Drop unmatched rows + clean temp columns
    # ══════════════════════════════════════════════════════════════════════════
    clean_cols = ['_norm_prod', '_norm_ing']
    matched_bridge = matched_bridge.drop(
        columns=[c for c in clean_cols if c in matched_bridge.columns]
    )

    # Verify ID format after remap
    pid_lens = matched_bridge['product_id'].astype(str).apply(len).unique()
    iid_lens = matched_bridge['ingredient_id'].dropna().astype(str).apply(len).unique()
    print(f"\n  Bridge ID format AFTER remap:")
    print(f"    product_id lengths    : {sorted(pid_lens)}-char {'✅' if all(l==16 for l in pid_lens) else '❌'}")
    print(f"    ingredient_id lengths : {sorted(iid_lens)}-char {'✅' if all(l==16 for l in iid_lens) else '❌'}")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 4 — Drop skincarisma products with no bridge rows
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "─" * 60)
    print("STEP 4: Drop skincarisma products with no bridge rows")
    print("─" * 60)

    bridge_prod_ids      = set(matched_bridge['product_id'].dropna().astype(str))
    prod_skinca_covered  = prod_skinca[prod_skinca['product_id'].isin(bridge_prod_ids)]
    prod_skinca_dropped  = prod_skinca[~prod_skinca['product_id'].isin(bridge_prod_ids)]

    # Rebuild product table
    prod_other   = prod[prod['source'] != 'skincarisma']
    prod_updated = pd.concat([prod_other, prod_skinca_covered], ignore_index=True)

    print(f"  Skincarisma products before : {len(prod_skinca):,}")
    print(f"  Dropped (no bridge rows)    : {len(prod_skinca_dropped):,}")
    print(f"  Remaining                   : {len(prod_skinca_covered):,}")
    print(f"\n  Product table before : {len(prod):,}")
    print(f"  Product table after  : {len(prod_updated):,}")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 5 — FK Integrity Check
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "─" * 60)
    print("STEP 5: FK Integrity Check — Skincarisma")
    print("─" * 60)

    prod_ids_skinca = set(prod_skinca_covered['product_id'].astype(str))
    ing_ids_skinca  = set(ing_skinca['ingredient_id'].astype(str))
    bridge_prod_ids_check = set(matched_bridge['product_id'].dropna().astype(str))
    bridge_ing_ids_check  = set(matched_bridge['ingredient_id'].dropna().astype(str))

    # Check 1: bridge.product_id → product table
    invalid_prod = matched_bridge[
        ~matched_bridge['product_id'].astype(str).isin(prod_ids_skinca)
    ]

    # Check 2: bridge.ingredient_id → ingredients table
    invalid_ing = matched_bridge[
        ~matched_bridge['ingredient_id'].fillna('').astype(str).isin(ing_ids_skinca)
    ]

    # Check 3: null IDs
    null_prod = matched_bridge['product_id'].isna().sum()
    null_ing  = matched_bridge['ingredient_id'].isna().sum()

    # Check 4: every skincarisma product has at least one bridge row
    unreferenced_prods = prod_skinca_covered[
        ~prod_skinca_covered['product_id'].astype(str).isin(bridge_prod_ids_check)
    ]

    # Check 5: unreferenced skincarisma ingredients (info only)
    unreferenced_ings = ing_skinca[
        ~ing_skinca['ingredient_id'].astype(str).isin(bridge_ing_ids_check)
    ]

    print(f"\n{'=' * 60}")
    print(f"FK INTEGRITY REPORT — SKINCARISMA")
    print(f"{'=' * 60}")

    print(f"\n  CHECK 1 — bridge.product_id → skincarisma products")
    print(f"    {'✅' if len(invalid_prod)==0 else '❌'} FK violations : {len(invalid_prod):,}")

    print(f"\n  CHECK 2 — bridge.ingredient_id → skincarisma ingredients")
    print(f"    {'✅' if len(invalid_ing)==0 else '❌'} FK violations : {len(invalid_ing):,}")

    print(f"\n  CHECK 3 — null IDs in bridge")
    print(f"    {'✅' if null_prod==0 else '❌'} Null product_id    : {null_prod:,}")
    print(f"    {'✅' if null_ing==0 else '❌'} Null ingredient_id : {null_ing:,}")

    print(f"\n  CHECK 4 — every skincarisma product has bridge rows")
    print(f"    {'✅' if len(unreferenced_prods)==0 else '❌'} Unreferenced products : {len(unreferenced_prods):,}")

    print(f"\n  CHECK 5 — skincarisma ingredient coverage (info only)")
    print(f"    ℹ️  Ingredients in table not referenced in bridge : {len(unreferenced_ings):,}")
    print(f"       (Not a FK violation — ingredients table can have more than bridge references)")

    print(f"\n  COVERAGE")
    cov_pct = 100 * len(prod_skinca_covered) / len(prod_skinca) if len(prod_skinca) > 0 else 0
    print(f"    Skincarisma products in bridge : {len(prod_skinca_covered):,}/{len(prod_skinca):,} ({cov_pct:.1f}%)")

    fk_ok = (
        len(invalid_prod) == 0 and
        len(invalid_ing)  == 0 and
        null_prod == 0 and
        null_ing  == 0 and
        len(unreferenced_prods) == 0
    )

    print(f"\n{'=' * 60}")
    print(f"  OVERALL : {'✅ FK INTEGRITY PASSED' if fk_ok else '❌ FK INTEGRITY FAILED'}")
    print(f"{'=' * 60}")

    # ── Save outputs ──────────────────────────────────────────────────────────
    matched_bridge.to_csv(OUTPUT_BRIDGE,   index=False)
    prod_updated.to_csv(OUTPUT_PRODUCTS,   index=False)

    if len(dropped_bridge) > 0:
        dropped_bridge.drop(columns=['_norm_prod'], errors='ignore').to_csv(
            OUTPUT_DROPPED_ROWS, index=False)

    if len(prod_skinca_dropped) > 0:
        prod_skinca_dropped[['product_id','product_name']].to_csv(
            OUTPUT_DROPPED_PRODUCTS, index=False)

    print(f"\nSaved:")
    print(f"  → {OUTPUT_BRIDGE:<40} ({len(matched_bridge):,} rows)")
    print(f"  → {OUTPUT_PRODUCTS:<40} ({len(prod_updated):,} rows)")
    if len(dropped_bridge) > 0:
        print(f"  → {OUTPUT_DROPPED_ROWS:<40} ({len(dropped_bridge):,} dropped bridge rows)")
    if len(prod_skinca_dropped) > 0:
        print(f"  → {OUTPUT_DROPPED_PRODUCTS:<40} ({len(prod_skinca_dropped):,} dropped products)")


if __name__ == "__main__":
    main()

Loading tables ...
  Products     : 24,900
  Ingredients  : 31,922
  Bridge       : 109,569

  Skincarisma products    : 4,380
  Skincarisma ingredients : 17,569

  Bridge ID format BEFORE remap:
    product_id lengths    : [np.int64(32)]-char
    ingredient_id lengths : [np.int64(32)]-char

────────────────────────────────────────────────────────────
STEP 1: Remapping product_id (32-char → 16-char)
────────────────────────────────────────────────────────────
  ✅ Remapped successfully : 106,201 rows
  ❌ No product match      : 3,368 rows  → dropped

────────────────────────────────────────────────────────────
STEP 2: Remapping ingredient_id (32-char → 16-char)
────────────────────────────────────────────────────────────
  ✅ Remapped successfully : 106,201 rows
  ❌ No ingredient match   : 0 rows

  Bridge ID format AFTER remap:
    product_id lengths    : [np.int64(16)]-char ✅
    ingredient_id lengths : [np.int64(16)]-char ✅

────────────────────────────────────────────────────────────

In [37]:
# audit 
import pandas as pd

# ── CONFIG ─────────────────────────────────────────────
PRODUCTS_FILE = "final_combinedlast.csv"
INGREDIENTS_FILE = "ingredients_updatedlast.csv"
BRIDGE_FILE = "bridge_skinca_final.csv"


# ── MAIN ───────────────────────────────────────────────
def audit():

    print("Loading files...")

    prod = pd.read_csv(PRODUCTS_FILE, low_memory=False)
    ing = pd.read_csv(INGREDIENTS_FILE, low_memory=False)
    bridge = pd.read_csv(BRIDGE_FILE, low_memory=False)

    print(f"Products    : {len(prod):,}")
    print(f"Ingredients : {len(ing):,}")
    print(f"Bridge      : {len(bridge):,}")

    errors = []

    print("\n" + "=" * 65)
    print("DATA MODEL AUDIT")
    print("=" * 65)

    # ────────────────────────────────────────────────────
    # 1. Required columns
    # ────────────────────────────────────────────────────
    print("\n1) REQUIRED COLUMNS")

    req_prod = {"product_id", "product_name"}
    req_ing = {"ingredient_id", "ingredient_name"}
    req_bridge = {"product_id", "ingredient_id"}

    for name, df, req in [
        ("Products", prod, req_prod),
        ("Ingredients", ing, req_ing),
        ("Bridge", bridge, req_bridge),
    ]:
        missing = req - set(df.columns)

        if missing:
            print(f"❌ {name} missing columns: {missing}")
            errors.append(f"{name} missing columns")
        else:
            print(f"✅ {name}")

    # ────────────────────────────────────────────────────
    # 2. Null IDs
    # ────────────────────────────────────────────────────
    print("\n2) NULL IDS")

    checks = [
        ("products.product_id", prod["product_id"].isna().sum()),
        ("ingredients.ingredient_id", ing["ingredient_id"].isna().sum()),
        ("bridge.product_id", bridge["product_id"].isna().sum()),
        ("bridge.ingredient_id", bridge["ingredient_id"].isna().sum()),
    ]

    for label, count in checks:
        if count == 0:
            print(f"✅ {label:<28} {count}")
        else:
            print(f"❌ {label:<28} {count}")
            errors.append(label)

    # ────────────────────────────────────────────────────
    # 3. Duplicate PKs
    # ────────────────────────────────────────────────────
    print("\n3) PRIMARY KEY DUPLICATES")

    prod_dup = prod["product_id"].duplicated().sum()
    ing_dup = ing["ingredient_id"].duplicated().sum()

    if prod_dup == 0:
        print("✅ products.product_id unique")
    else:
        print(f"❌ duplicate product_id: {prod_dup:,}")
        errors.append("duplicate product_id")

    if ing_dup == 0:
        print("✅ ingredients.ingredient_id unique")
    else:
        print(f"❌ duplicate ingredient_id: {ing_dup:,}")
        errors.append("duplicate ingredient_id")

    # ────────────────────────────────────────────────────
    # 4. FK product
    # ────────────────────────────────────────────────────
    print("\n4) FK bridge.product_id → products")

    prod_ids = set(prod["product_id"].astype(str))
    invalid_prod = bridge[
        ~bridge["product_id"].astype(str).isin(prod_ids)
    ]

    if len(invalid_prod) == 0:
        print("✅ all bridge product_id valid")
    else:
        print(f"❌ invalid bridge product refs: {len(invalid_prod):,}")
        errors.append("invalid product refs")

    # ────────────────────────────────────────────────────
    # 5. FK ingredient
    # ────────────────────────────────────────────────────
    print("\n5) FK bridge.ingredient_id → ingredients")

    ing_ids = set(ing["ingredient_id"].astype(str))
    invalid_ing = bridge[
        ~bridge["ingredient_id"].astype(str).isin(ing_ids)
    ]

    if len(invalid_ing) == 0:
        print("✅ all bridge ingredient_id valid")
    else:
        print(f"❌ invalid bridge ingredient refs: {len(invalid_ing):,}")
        errors.append("invalid ingredient refs")

    # ────────────────────────────────────────────────────
    # 6. Duplicate bridge pairs
    # ────────────────────────────────────────────────────
    print("\n6) DUPLICATE BRIDGE PAIRS")

    dup_pairs = bridge.duplicated(
        subset=["product_id", "ingredient_id"]
    ).sum()

    if dup_pairs == 0:
        print("✅ bridge pairs unique")
    else:
        print(f"❌ duplicate bridge pairs: {dup_pairs:,}")
        errors.append("duplicate bridge pairs")

    # ────────────────────────────────────────────────────
    # 7. Product coverage
    # ────────────────────────────────────────────────────
    print("\n7) PRODUCT COVERAGE")

    bridge_prod_ids = set(bridge["product_id"].astype(str))

    orphan_products = prod[
        ~prod["product_id"].astype(str).isin(bridge_prod_ids)
    ]

    print(
        f"ℹ️ products not referenced: {len(orphan_products):,}"
    )

    # ────────────────────────────────────────────────────
    # 8. Ingredient coverage
    # ────────────────────────────────────────────────────
    print("\n8) INGREDIENT COVERAGE")

    bridge_ing_ids = set(bridge["ingredient_id"].astype(str))

    orphan_ingredients = ing[
        ~ing["ingredient_id"].astype(str).isin(bridge_ing_ids)
    ]

    print(
        f"ℹ️ ingredients not referenced: {len(orphan_ingredients):,}"
    )

    # ────────────────────────────────────────────────────
    # 9. ID length consistency
    # ────────────────────────────────────────────────────
    print("\n9) ID LENGTHS")

    for label, series in [
        ("products.product_id", prod["product_id"]),
        ("ingredients.ingredient_id", ing["ingredient_id"]),
        ("bridge.product_id", bridge["product_id"]),
        ("bridge.ingredient_id", bridge["ingredient_id"]),
    ]:

        lengths = sorted(
            series.dropna()
            .astype(str)
            .apply(len)
            .unique()
        )

        print(f"{label:<30} {lengths}")

    # ────────────────────────────────────────────────────
    # FINAL
    # ────────────────────────────────────────────────────
    print("\n" + "=" * 65)

    if len(errors) == 0:
        print("✅ OVERALL: PASSED")
        print("Safe for Spark / PostgreSQL / Snowflake / Power BI")
    else:
        print("❌ OVERALL: FAILED")
        print("Problems found:")
        for e in errors:
            print("-", e)

    print("=" * 65)


if __name__ == "__main__":
    audit()

Loading files...
Products    : 24,109
Ingredients : 31,922
Bridge      : 106,201

DATA MODEL AUDIT

1) REQUIRED COLUMNS
✅ Products
✅ Ingredients
✅ Bridge

2) NULL IDS
✅ products.product_id          0
✅ ingredients.ingredient_id    0
✅ bridge.product_id            0
✅ bridge.ingredient_id         0

3) PRIMARY KEY DUPLICATES
✅ products.product_id unique
✅ ingredients.ingredient_id unique

4) FK bridge.product_id → products
✅ all bridge product_id valid

5) FK bridge.ingredient_id → ingredients
✅ all bridge ingredient_id valid

6) DUPLICATE BRIDGE PAIRS
❌ duplicate bridge pairs: 394

7) PRODUCT COVERAGE
ℹ️ products not referenced: 20,520

8) INGREDIENT COVERAGE
ℹ️ ingredients not referenced: 14,728

9) ID LENGTHS
products.product_id            [np.int64(16)]
ingredients.ingredient_id      [np.int64(16)]
bridge.product_id              [np.int64(16)]
bridge.ingredient_id           [np.int64(16)]

❌ OVERALL: FAILED
Problems found:
- duplicate bridge pairs


In [38]:
# duplicated in the  "bridge_skinca_final.csv"
# check duplicates
b=pd.read_csv('bridge_skinca_final.csv')
dupes = b[
    b.duplicated(
        subset=["product_id", "ingredient_id"],
        keep=False
    )
].sort_values(["product_id", "ingredient_id"])

print(dupes.head(20))
print("Duplicate rows:", len(dupes))

             product_id    product_name     ingredient_id  \
11392  08a5a58716e2fd07   Aloe Vera Gel  12c5a85f2da191a6   
11415  08a5a58716e2fd07   Aloe Vera Gel  12c5a85f2da191a6   
11384  08a5a58716e2fd07   Aloe Vera Gel  3fb665e57194cfd6   
11418  08a5a58716e2fd07   Aloe Vera Gel  3fb665e57194cfd6   
11427  08a5a58716e2fd07   Aloe Vera Gel  3fb665e57194cfd6   
11391  08a5a58716e2fd07   Aloe Vera Gel  450c26b988b881e6   
11444  08a5a58716e2fd07   Aloe Vera Gel  450c26b988b881e6   
11417  08a5a58716e2fd07   Aloe Vera Gel  5cc09d76635a757a   
11442  08a5a58716e2fd07   Aloe Vera Gel  5cc09d76635a757a   
11386  08a5a58716e2fd07   Aloe Vera Gel  aa4d891cf42af9ed   
11433  08a5a58716e2fd07   Aloe Vera Gel  aa4d891cf42af9ed   
11403  08a5a58716e2fd07   Aloe Vera Gel  ade7c79e85497153   
11414  08a5a58716e2fd07   Aloe Vera Gel  ade7c79e85497153   
11443  08a5a58716e2fd07   Aloe Vera Gel  ade7c79e85497153   
11393  08a5a58716e2fd07   Aloe Vera Gel  f528bea8e5c67d52   
11424  08a5a58716e2fd07 

In [39]:
# clean duplicated that has less information (more null) and keep rich rows in bridge clean (skincarisma)
import pandas as pd

# =========================================================
# CONFIG
# =========================================================
BRIDGE_FILE = "bridge_skinca_final.csv"
OUTPUT_FILE = "bridge_skinca_final_clean.csv"


# =========================================================
# LOAD
# =========================================================
print("Loading bridge...")

b = pd.read_csv(BRIDGE_FILE, low_memory=False)

print(f"Rows loaded: {len(b):,}")


# =========================================================
# DUPLICATE CHECK
# =========================================================
dup_mask = b.duplicated(
    subset=["product_id", "ingredient_id"],
    keep=False
)

dup_rows = b[dup_mask].copy()

print("\n" + "=" * 65)
print("DUPLICATE ANALYSIS")
print("=" * 65)

print(
    "Rows involved in duplicate groups:",
    f"{len(dup_rows):,}"
)

print(
    "Duplicate product+ingredient pairs:",
    f"{dup_rows[['product_id','ingredient_id']].drop_duplicates().shape[0]:,}"
)


# =========================================================
# SCORING
# higher = better
# =========================================================
b["_non_null_count"] = b.notna().sum(axis=1)

b["_score"] = (
    b["_non_null_count"]
    + b["ingredient_concentration"].notna().astype(int) * 3
    + b["evidence_level"].notna().astype(int) * 2
    + b["raw_ingredient_name"].notna().astype(int) * 1
)

# optional:
# smaller ingredient_position preferred
# if null -> very large
b["_position_rank"] = (
    pd.to_numeric(
        b["ingredient_position"],
        errors="coerce"
    )
    .fillna(999999)
)


# =========================================================
# SORT BEST ROW FIRST
# =========================================================
b_sorted = b.sort_values(
    [
        "product_id",
        "ingredient_id",
        "_score",
        "_position_rank",
    ],
    ascending=[True, True, False, True],
)


# =========================================================
# KEEP BEST ROW
# =========================================================
bridge_clean = (
    b_sorted
    .drop_duplicates(
        subset=["product_id", "ingredient_id"],
        keep="first"
    )
    .drop(
        columns=[
            "_non_null_count",
            "_score",
            "_position_rank",
        ]
    )
    .copy()
)


# =========================================================
# SAVE
# =========================================================
bridge_clean.to_csv(
    OUTPUT_FILE,
    index=False
)


# =========================================================
# PRINT RESULTS
# =========================================================
print("\n" + "=" * 65)
print("CLEANING RESULTS")
print("=" * 65)

print(f"Before rows : {len(b):,}")
print(f"After rows  : {len(bridge_clean):,}")
print(f"Removed     : {len(b) - len(bridge_clean):,}")

print()

print(
    "Unique product_id:",
    f"{bridge_clean['product_id'].nunique():,}"
)

print(
    "Unique ingredient_id:",
    f"{bridge_clean['ingredient_id'].nunique():,}"
)

remaining_dupes = bridge_clean.duplicated(
    subset=["product_id", "ingredient_id"]
).sum()

print(
    "Remaining duplicates:",
    remaining_dupes
)

print()

print("Saved clean bridge:")
print(OUTPUT_FILE)

print("\nSample of cleaned data:")
print(
    bridge_clean.head(10)
)

print("\n" + "=" * 65)

if remaining_dupes == 0:
    print("✅ CLEAN SUCCESSFUL")
    print("Bridge is unique on product_id + ingredient_id")
else:
    print("❌ duplicates still exist")

print("=" * 65)

Loading bridge...
Rows loaded: 106,201

DUPLICATE ANALYSIS
Rows involved in duplicate groups: 708
Duplicate product+ingredient pairs: 314

CLEANING RESULTS
Before rows : 106,201
After rows  : 105,807
Removed     : 394

Unique product_id: 3,589
Unique ingredient_id: 17,194
Remaining duplicates: 0

Saved clean bridge:
bridge_skinca_final_clean.csv

Sample of cleaned data:
             product_id         product_name     ingredient_id  \
34429  002333f784158dd4  Cicaful Calming Gel  0cb72f5dc9697b56   
34419  002333f784158dd4  Cicaful Calming Gel  1808d6fa4f98120b   
34427  002333f784158dd4  Cicaful Calming Gel  1d8343ce68d08e1c   
34431  002333f784158dd4  Cicaful Calming Gel  2d986f412499c362   
34424  002333f784158dd4  Cicaful Calming Gel  380227760d45b95b   
34433  002333f784158dd4  Cicaful Calming Gel  706a31cabc513b1a   
34430  002333f784158dd4  Cicaful Calming Gel  9460370bb0ca1c98   
34434  002333f784158dd4  Cicaful Calming Gel  a20e6203f4ac77cd   
34420  002333f784158dd4  Cicaful 

In [40]:
# check full fk integrity after remvoing duplicated 
import pandas as pd

# =========================================================
# CONFIG
# =========================================================
PRODUCTS_FILE = "final_combinedlast.csv"
INGREDIENTS_FILE = "ingredients_updatedlast.csv"
BRIDGE_FILE = "bridge_skinca_final_clean.csv"


# =========================================================
# LOAD
# =========================================================
print("Loading files...")

prod = pd.read_csv(PRODUCTS_FILE, low_memory=False)
ing = pd.read_csv(INGREDIENTS_FILE, low_memory=False)
bridge = pd.read_csv(BRIDGE_FILE, low_memory=False)

print(f"Products    : {len(prod):,}")
print(f"Ingredients : {len(ing):,}")
print(f"Bridge      : {len(bridge):,}")


errors = []

print("\n" + "=" * 70)
print("FULL DATA MODEL AUDIT")
print("=" * 70)


# =========================================================
# 1 REQUIRED COLUMNS
# =========================================================
print("\n1) REQUIRED COLUMNS")

required = {
    "Products": (
        prod,
        {"product_id", "product_name"},
    ),
    "Ingredients": (
        ing,
        {"ingredient_id", "ingredient_name"},
    ),
    "Bridge": (
        bridge,
        {"product_id", "ingredient_id"},
    ),
}

for name, (df, cols) in required.items():

    missing = cols - set(df.columns)

    if missing:
        print(f"❌ {name}: missing {missing}")
        errors.append(f"{name} missing columns")
    else:
        print(f"✅ {name}")


# =========================================================
# 2 NULL IDS
# =========================================================
print("\n2) NULL IDS")

null_checks = {
    "products.product_id":
        prod["product_id"].isna().sum(),

    "ingredients.ingredient_id":
        ing["ingredient_id"].isna().sum(),

    "bridge.product_id":
        bridge["product_id"].isna().sum(),

    "bridge.ingredient_id":
        bridge["ingredient_id"].isna().sum(),
}

for label, cnt in null_checks.items():

    if cnt == 0:
        print(f"✅ {label:<30} {cnt}")
    else:
        print(f"❌ {label:<30} {cnt}")
        errors.append(label)


# =========================================================
# 3 PK UNIQUENESS
# =========================================================
print("\n3) PRIMARY KEY UNIQUENESS")

prod_dup = prod["product_id"].duplicated().sum()
ing_dup = ing["ingredient_id"].duplicated().sum()

if prod_dup == 0:
    print("✅ products.product_id unique")
else:
    print(f"❌ duplicate product_id: {prod_dup:,}")
    errors.append("duplicate product_id")

if ing_dup == 0:
    print("✅ ingredients.ingredient_id unique")
else:
    print(f"❌ duplicate ingredient_id: {ing_dup:,}")
    errors.append("duplicate ingredient_id")


# =========================================================
# 4 BRIDGE UNIQUENESS
# =========================================================
print("\n4) BRIDGE UNIQUE PAIRS")

bridge_dup = bridge.duplicated(
    subset=["product_id", "ingredient_id"]
).sum()

if bridge_dup == 0:
    print("✅ bridge unique on product_id + ingredient_id")
else:
    print(f"❌ duplicate bridge pairs: {bridge_dup:,}")
    errors.append("duplicate bridge pairs")


# =========================================================
# 5 FK PRODUCT
# =========================================================
print("\n5) FK bridge.product_id → products")

prod_ids = set(prod["product_id"].astype(str))

invalid_prod = bridge[
    ~bridge["product_id"]
    .astype(str)
    .isin(prod_ids)
]

if len(invalid_prod) == 0:
    print("✅ all bridge product refs valid")
else:
    print(
        "❌ invalid bridge product refs:",
        len(invalid_prod)
    )
    print(
        invalid_prod[
            ["product_id"]
        ].drop_duplicates().head(10)
    )
    errors.append("invalid product refs")


# =========================================================
# 6 FK INGREDIENT
# =========================================================
print("\n6) FK bridge.ingredient_id → ingredients")

ing_ids = set(ing["ingredient_id"].astype(str))

invalid_ing = bridge[
    ~bridge["ingredient_id"]
    .astype(str)
    .isin(ing_ids)
]

if len(invalid_ing) == 0:
    print("✅ all bridge ingredient refs valid")
else:
    print(
        "❌ invalid bridge ingredient refs:",
        len(invalid_ing)
    )
    print(
        invalid_ing[
            ["ingredient_id"]
        ].drop_duplicates().head(10)
    )
    errors.append("invalid ingredient refs")


# =========================================================
# 7 PRODUCT COVERAGE
# =========================================================
print("\n7) PRODUCT COVERAGE (info)")

bridge_prod_ids = set(
    bridge["product_id"].astype(str)
)

unref_products = prod[
    ~prod["product_id"]
    .astype(str)
    .isin(bridge_prod_ids)
]

print(
    "ℹ️ products not referenced:",
    f"{len(unref_products):,}"
)

print(
    "ℹ️ products referenced:",
    f"{bridge['product_id'].nunique():,}"
)


# =========================================================
# 8 INGREDIENT COVERAGE
# =========================================================
print("\n8) INGREDIENT COVERAGE (info)")

bridge_ing_ids = set(
    bridge["ingredient_id"].astype(str)
)

unref_ing = ing[
    ~ing["ingredient_id"]
    .astype(str)
    .isin(bridge_ing_ids)
]

print(
    "ℹ️ ingredients not referenced:",
    f"{len(unref_ing):,}"
)

print(
    "ℹ️ ingredients referenced:",
    f"{bridge['ingredient_id'].nunique():,}"
)


# =========================================================
# 9 SUMMARY COUNTS
# =========================================================
print("\n9) SUMMARY COUNTS")

print(
    "Products total         :",
    f"{len(prod):,}"
)

print(
    "Ingredients total      :",
    f"{len(ing):,}"
)

print(
    "Bridge total           :",
    f"{len(bridge):,}"
)


# =========================================================
# FINAL
# =========================================================
print("\n" + "=" * 70)

if len(errors) == 0:
    print("✅ OVERALL: PASSED")
    print(
        "Safe for Spark / PostgreSQL / Snowflake / Power BI"
    )
else:
    print("❌ OVERALL: FAILED")
    print("Problems found:")

    for e in errors:
        print("-", e)

print("=" * 70)

Loading files...
Products    : 24,109
Ingredients : 31,922
Bridge      : 105,807

FULL DATA MODEL AUDIT

1) REQUIRED COLUMNS
✅ Products
✅ Ingredients
✅ Bridge

2) NULL IDS
✅ products.product_id            0
✅ ingredients.ingredient_id      0
✅ bridge.product_id              0
✅ bridge.ingredient_id           0

3) PRIMARY KEY UNIQUENESS
✅ products.product_id unique
✅ ingredients.ingredient_id unique

4) BRIDGE UNIQUE PAIRS
✅ bridge unique on product_id + ingredient_id

5) FK bridge.product_id → products
✅ all bridge product refs valid

6) FK bridge.ingredient_id → ingredients
✅ all bridge ingredient refs valid

7) PRODUCT COVERAGE (info)
ℹ️ products not referenced: 20,520
ℹ️ products referenced: 3,589

8) INGREDIENT COVERAGE (info)
ℹ️ ingredients not referenced: 14,728
ℹ️ ingredients referenced: 17,194

9) SUMMARY COUNTS
Products total         : 24,109
Ingredients total      : 31,922
Bridge total           : 105,807

✅ OVERALL: PASSED
Safe for Spark / PostgreSQL / Snowflake / Power BI


In [41]:
# other product and update bridge table

"""
Build Bridge for Non-Skincarisma Products & Full FK Audit
==========================================================
Run this AFTER the skincarisma bridge script.

This script:
  1. Takes all non-skincarisma products from the product table
  2. Parses their ingredients_list column
  3. Filters garbage strings
  4. Matches each ingredient name to the ingredients table (by normalized name)
  5. Builds bridge rows with correct 16-char product_id and ingredient_id
  6. Combines with the clean skincarisma bridge
  7. Removes duplicate (product_id, ingredient_id) pairs — keeps richest row
  8. Runs full FK audit across ALL products and ALL ingredients

Inputs:
  - PRODUCTS_FILE      → final_combinedlast.csv       (all sources, 16-char IDs)
  - INGREDIENTS_FILE   → ingredients_updatedlast.csv  (all sources, 16-char IDs)
  - SKINCA_BRIDGE_FILE → bridge_skinca_final_clean.csv (skincarisma bridge, clean)

Outputs:
  - bridge_final.csv                  → full combined bridge (all sources)
  - bridge_unmatched_ingredients.csv  → ingredient names not found in table
  - bridge_products_no_bridge.csv     → products with no bridge rows
"""

import pandas as pd
import ast
import re
import unicodedata

# ── CONFIG ────────────────────────────────────────────────────────────────────
PRODUCTS_FILE      = "final_combinedlast.csv"
INGREDIENTS_FILE   = "ingredients_updatedlast.csv"
SKINCA_BRIDGE_FILE = "bridge_skinca_final_clean.csv"

OUTPUT_BRIDGE        = "bridge_finallast.csv"
OUTPUT_UNMATCHED_ING = "bridge_unmatched_ingredients.csv"
OUTPUT_NO_BRIDGE     = "bridge_products_no_bridge.csv"

SKINCARISMA_SOURCE = "skincarisma"


# ── GARBAGE FILTER ────────────────────────────────────────────────────────────
GARBAGE_PATTERNS = [
    r'^#',                   # Excel errors: #NAME?, #REF!
    r'ingredients\s*:',      # Label strings
    r'étape\s*\d',           # French step labels
    r'step\s*\d',            # English step labels
    r'^\(\*',                # annotation labels
    r'^-\s*exfoliant',
    r'^\d+$',                # pure numbers
    r'^[^a-zA-Z\(\*]',      # starts with non-letter
]
GARBAGE_RE = re.compile('|'.join(GARBAGE_PATTERNS), re.IGNORECASE)


# ── HELPERS ───────────────────────────────────────────────────────────────────

def normalize(name) -> str:
    """Unicode normalization + lowercase + strip."""
    if pd.isna(name):
        return ''
    name = unicodedata.normalize('NFKD', str(name))
    name = ''.join(c for c in name if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', name.lower().strip())


def is_valid_ingredient(name: str) -> bool:
    """Filter out garbage strings."""
    name = name.strip()
    if len(name) < 3:             return False
    if GARBAGE_RE.search(name):   return False
    if name.count(' ') > 15:      return False
    return True


def parse_ingredients(val) -> list:
    """Parse ingredients_list cell → list of ingredient name strings."""
    if pd.isna(val):
        return []
    s = str(val).strip()
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return [str(i).strip() for i in parsed if str(i).strip()]
    except Exception:
        pass
    return [i.strip().strip("'\"") for i in re.split(r',(?=\s)', s) if i.strip()]


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():

    # ── Load ──────────────────────────────────────────────────────────────────
    print("Loading tables ...")
    prod   = pd.read_csv(PRODUCTS_FILE,      low_memory=False)
    ing    = pd.read_csv(INGREDIENTS_FILE,   low_memory=False)
    bridge = pd.read_csv(SKINCA_BRIDGE_FILE, low_memory=False)

    print(f"  Products (all sources)    : {len(prod):,}")
    print(f"  Ingredients (all sources) : {len(ing):,}")
    print(f"  Skincarisma bridge (clean): {len(bridge):,}")

    print(f"\n  Product sources:")
    for source, count in prod['source'].value_counts().items():
        tag = '⟵ already covered by skincarisma bridge' if source == SKINCARISMA_SOURCE else '⟵ to process now'
        print(f"    {source:<25}: {count:,}  {tag}")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 1 — Filter to non-skincarisma products
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("STEP 1: Filter non-skincarisma products")
    print("=" * 65)

    prod_other = prod[prod['source'] != SKINCARISMA_SOURCE].copy()
    has_list   = prod_other[prod_other['ingredients_list'].notna()]
    no_list    = prod_other[prod_other['ingredients_list'].isna()]

    print(f"\n  Non-skincarisma products total   : {len(prod_other):,}")
    print(f"    → With ingredients_list         : {len(has_list):,}")
    print(f"    → Without ingredients_list      : {len(no_list):,}  (skipped — no data)")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 2 — Build ingredient name lookup from ingredients table
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("STEP 2: Build ingredient lookup from ingredients table")
    print("=" * 65)

    ing['_norm'] = ing['ingredient_name'].apply(normalize)

    # norm_name → (ingredient_id, canonical_ingredient_name)
    ing_lookup = {
        row['_norm']: (row['ingredient_id'], row['ingredient_name'])
        for _, row in ing.iterrows()
    }

    print(f"\n  Ingredient lookup entries: {len(ing_lookup):,}")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 3 — Build bridge rows for non-skincarisma products
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("STEP 3: Build bridge rows for non-skincarisma products")
    print("=" * 65)

    new_rows       = []
    unmatched_rows = []
    covered_prods  = set()

    for _, row in has_list.iterrows():
        ingredients = parse_ingredients(row['ingredients_list'])
        valid_ings  = [
            i.strip() for i in ingredients
            if i.strip() and is_valid_ingredient(i.strip())
        ]

        found_any = False

        for pos, ing_name in enumerate(valid_ings, start=1):
            match = ing_lookup.get(normalize(ing_name))

            if match is None:
                # Ingredient not found in table — log for review
                unmatched_rows.append({
                    'product_id'         : row['product_id'],
                    'product_name'       : row['product_name'],
                    'source'             : row['source'],
                    'raw_ingredient_name': ing_name,
                })
            else:
                ing_id, ing_canonical = match
                new_rows.append({
                    'product_id'              : row['product_id'],
                    'product_name'            : row['product_name'],
                    'ingredient_id'           : ing_id,
                    'ingredient_name'         : ing_canonical,
                    'ingredient_position'     : pos,
                    'ingredient_concentration': None,
                    'raw_ingredient_name'     : ing_name,
                    'evidence_level'          : None,
                    'source'                  : row['source'],
                })
                found_any = True

        if found_any:
            covered_prods.add(row['product_id'])

    new_bridge = pd.DataFrame(new_rows)

    print(f"\n  New bridge rows created          : {len(new_bridge):,}")
    print(f"  Non-skinca products with bridge  : {len(covered_prods):,}")
    print(f"  Non-skinca products no match     : {len(has_list) - len(covered_prods):,}")
    print(f"  Unmatched ingredient names       : {len(unmatched_rows):,}")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 4 — Combine skincarisma bridge + new rows
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("STEP 4: Combine bridges")
    print("=" * 65)

    combined = pd.concat([bridge, new_bridge], ignore_index=True)

    print(f"\n  Skincarisma rows  : {len(bridge):,}")
    print(f"  New rows added    : {len(new_bridge):,}")
    print(f"  Combined total    : {len(combined):,}")

    print(f"\n  By source:")
    for source, count in combined['source'].value_counts().items():
        print(f"    {source:<25}: {count:,}")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 5 — Deduplicate (product_id, ingredient_id) pairs
    #          Keep richest row (most non-null fields)
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("STEP 5: Remove duplicate (product_id, ingredient_id) pairs")
    print("=" * 65)

    dup_before = combined.duplicated(subset=['product_id', 'ingredient_id']).sum()
    print(f"\n  Duplicate pairs before dedup: {dup_before:,}")

    if dup_before > 0:
        # Score rows — higher = richer
        combined['_non_null'] = combined.notna().sum(axis=1)
        combined['_score'] = (
            combined['_non_null']
            + combined['ingredient_concentration'].notna().astype(int) * 3
            + combined['evidence_level'].notna().astype(int) * 2
            + combined['raw_ingredient_name'].notna().astype(int) * 1
        )
        combined['_pos_rank'] = (
            pd.to_numeric(combined['ingredient_position'], errors='coerce').fillna(999999)
        )

        # Sort best first, keep first per pair
        combined = combined.sort_values(
            ['product_id', 'ingredient_id', '_score', '_pos_rank'],
            ascending=[True, True, False, True]
        )
        combined = combined.drop_duplicates(
            subset=['product_id', 'ingredient_id'], keep='first'
        )
        combined = combined.drop(columns=['_non_null', '_score', '_pos_rank'])

        dup_after = combined.duplicated(subset=['product_id', 'ingredient_id']).sum()
        print(f"  Duplicate pairs after dedup  : {dup_after:,}")
        print(f"  Rows removed                 : {dup_before:,}")

    # ══════════════════════════════════════════════════════════════════════════
    # STEP 6 — Full FK Audit (same audit logic from your notebook)
    # ══════════════════════════════════════════════════════════════════════════
    print("\n" + "=" * 70)
    print("FULL DATA MODEL AUDIT — ALL PRODUCTS + ALL INGREDIENTS")
    print("=" * 70)

    errors = []

    prod_ids       = set(prod['product_id'].astype(str))
    ing_ids        = set(ing['ingredient_id'].astype(str))
    bridge_prod_ids = set(combined['product_id'].astype(str))
    bridge_ing_ids  = set(combined['ingredient_id'].astype(str))

    # 1. Required columns
    print("\n1) REQUIRED COLUMNS")
    for name, df, req in [
        ("Products",    prod,     {"product_id", "product_name"}),
        ("Ingredients", ing,      {"ingredient_id", "ingredient_name"}),
        ("Bridge",      combined, {"product_id", "ingredient_id"}),
    ]:
        missing = req - set(df.columns)
        if missing:
            print(f"  ❌ {name}: missing {missing}")
            errors.append(f"{name} missing columns")
        else:
            print(f"  ✅ {name}")

    # 2. Null IDs
    print("\n2) NULL IDS")
    null_checks = {
        "products.product_id"      : prod["product_id"].isna().sum(),
        "ingredients.ingredient_id": ing["ingredient_id"].isna().sum(),
        "bridge.product_id"        : combined["product_id"].isna().sum(),
        "bridge.ingredient_id"     : combined["ingredient_id"].isna().sum(),
    }
    for label, cnt in null_checks.items():
        icon = '✅' if cnt == 0 else '❌'
        print(f"  {icon} {label:<32} : {cnt:,}")
        if cnt > 0:
            errors.append(f"null {label}")

    # 3. PK uniqueness
    print("\n3) PRIMARY KEY UNIQUENESS")
    prod_dup = prod["product_id"].duplicated().sum()
    ing_dup  = ing["ingredient_id"].duplicated().sum()
    for label, dup in [("products.product_id", prod_dup), ("ingredients.ingredient_id", ing_dup)]:
        icon = '✅' if dup == 0 else '❌'
        print(f"  {icon} {label}: {dup:,} duplicates")
        if dup > 0:
            errors.append(f"duplicate {label}")

    # 4. Bridge unique pairs
    print("\n4) BRIDGE UNIQUE PAIRS")
    bridge_dup = combined.duplicated(subset=["product_id", "ingredient_id"]).sum()
    icon = '✅' if bridge_dup == 0 else '❌'
    print(f"  {icon} bridge unique on product_id + ingredient_id: {bridge_dup:,} duplicates")
    if bridge_dup > 0:
        errors.append("duplicate bridge pairs")

    # 5. FK product
    print("\n5) FK bridge.product_id → products")
    invalid_prod = combined[~combined["product_id"].astype(str).isin(prod_ids)]
    icon = '✅' if len(invalid_prod) == 0 else '❌'
    print(f"  {icon} invalid bridge product refs: {len(invalid_prod):,}")
    if len(invalid_prod) > 0:
        errors.append("invalid product refs")

    # 6. FK ingredient
    print("\n6) FK bridge.ingredient_id → ingredients")
    invalid_ing = combined[~combined["ingredient_id"].astype(str).isin(ing_ids)]
    icon = '✅' if len(invalid_ing) == 0 else '❌'
    print(f"  {icon} invalid bridge ingredient refs: {len(invalid_ing):,}")
    if len(invalid_ing) > 0:
        errors.append("invalid ingredient refs")

    # 7. Product coverage
    print("\n7) PRODUCT COVERAGE")
    unref_prods = prod[~prod["product_id"].astype(str).isin(bridge_prod_ids)]
    has_no_list = unref_prods[unref_prods['ingredients_list'].isna()] if 'ingredients_list' in unref_prods.columns else pd.DataFrame()
    has_list_no_bridge = unref_prods[unref_prods['ingredients_list'].notna()] if 'ingredients_list' in unref_prods.columns else pd.DataFrame()

    print(f"  ✅ Products with bridge rows     : {len(prod) - len(unref_prods):,}/{len(prod):,}")
    print(f"  ℹ️  Products without bridge rows : {len(unref_prods):,}")
    print(f"     → No ingredients_list (no data)  : {len(has_no_list):,}")
    print(f"     → Has list but no match found    : {len(has_list_no_bridge):,}")
    print(f"\n  By source:")
    for source, grp in prod.groupby('source'):
        covered = grp[grp['product_id'].astype(str).isin(bridge_prod_ids)]
        pct     = 100 * len(covered) / len(grp)
        missing = len(grp) - len(covered)
        print(f"    {source:<25}: {len(covered):>6,}/{len(grp):>6,} ({pct:.1f}%)  | {missing:>5,} no bridge")

    # 8. Ingredient coverage
    print("\n8) INGREDIENT COVERAGE")
    unref_ings = ing[~ing["ingredient_id"].astype(str).isin(bridge_ing_ids)]
    print(f"  ℹ️  Ingredients referenced     : {ing['ingredient_id'].nunique() - len(unref_ings):,}/{len(ing):,}")
    print(f"  ℹ️  Ingredients not referenced : {len(unref_ings):,}  (not a violation)")

    # 9. ID lengths
    print("\n9) ID LENGTHS")
    for label, series in [
        ("products.product_id",       prod["product_id"]),
        ("ingredients.ingredient_id", ing["ingredient_id"]),
        ("bridge.product_id",         combined["product_id"]),
        ("bridge.ingredient_id",      combined["ingredient_id"]),
    ]:
        lengths = sorted(series.dropna().astype(str).apply(len).unique())
        all_16 = all(l == 16 for l in lengths)
        icon = '✅' if all_16 else '⚠️'
        print(f"  {icon} {label:<32}: {lengths}")

    # 10. Summary counts
    print("\n10) SUMMARY COUNTS")
    print(f"  Products    total : {len(prod):,}")
    print(f"  Ingredients total : {len(ing):,}")
    print(f"  Bridge      total : {len(combined):,}")

    # Final verdict
    print("\n" + "=" * 70)
    if len(errors) == 0:
        print("✅ OVERALL: PASSED")
        print("Safe for Spark / PostgreSQL / Snowflake / Power BI")
    else:
        print("❌ OVERALL: FAILED")
        print("Problems found:")
        for e in errors:
            print(f"  - {e}")
    print("=" * 70)

    # ── Save outputs ──────────────────────────────────────────────────────────
    combined.to_csv(OUTPUT_BRIDGE, index=False)
    print(f"\nSaved:")
    print(f"  → {OUTPUT_BRIDGE:<45} ({len(combined):,} rows)")

    if unmatched_rows:
        pd.DataFrame(unmatched_rows).to_csv(OUTPUT_UNMATCHED_ING, index=False)
        print(f"  → {OUTPUT_UNMATCHED_ING:<45} ({len(unmatched_rows):,} rows)")

    if len(unref_prods) > 0:
        unref_prods[['product_id', 'product_name', 'source', 'ingredients_list']].to_csv(
            OUTPUT_NO_BRIDGE, index=False)
        print(f"  → {OUTPUT_NO_BRIDGE:<45} ({len(unref_prods):,} products)")


if __name__ == "__main__":
    main()

Loading tables ...
  Products (all sources)    : 24,109
  Ingredients (all sources) : 31,922
  Skincarisma bridge (clean): 105,807

  Product sources:
    datasheet                : 15,804  ⟵ to process now
    skincarisma              : 3,589  ⟵ already covered by skincarisma bridge
    dermstore                : 2,377  ⟵ to process now
    cosmetics                : 1,304  ⟵ to process now
    skincare_products        : 1,035  ⟵ to process now

STEP 1: Filter non-skincarisma products

  Non-skincarisma products total   : 20,520
    → With ingredients_list         : 20,520
    → Without ingredients_list      : 0  (skipped — no data)

STEP 2: Build ingredient lookup from ingredients table

  Ingredient lookup entries: 31,869

STEP 3: Build bridge rows for non-skincarisma products

  New bridge rows created          : 551,229
  Non-skinca products with bridge  : 20,026
  Non-skinca products no match     : 494
  Unmatched ingredient names       : 0

STEP 4: Combine bridges

  Skincarisma

In [42]:
# This is the final audit 
"""
Final Data Model Audit
=======================
Complete integrity check across all three tables:
  - products        (final_combinedlast.csv)
  - ingredients     (ingredients_updatedlast.csv)
  - bridge          (bridge_finallast.csv)

Checks:
  1.  Required columns exist
  2.  Null IDs
  3.  PK uniqueness (products + ingredients)
  4.  Bridge unique pairs (product_id + ingredient_id)
  5.  FK bridge.product_id    → products
  6.  FK bridge.ingredient_id → ingredients
  7.  ID format (all must be 16-char)
  8.  ID type consistency (no mixed int/str)
  9.  Product coverage (by source)
  10. Ingredient coverage
  11. Ingredient name consistency (bridge vs ingredients table)
  12. Product name consistency (bridge vs products table)
  13. Source column check (no nulls, known values only)
  14. Bridge composition by source
  15. Overall summary counts

Outputs:
  - audit_report.txt                   → full report saved to file
  - audit_invalid_product_refs.csv     → FK violations on product_id  (if any)
  - audit_invalid_ingredient_refs.csv  → FK violations on ingredient_id (if any)
  - audit_name_mismatches.csv          → name mismatches bridge vs tables (if any)
  - audit_products_no_bridge.csv       → products with no bridge rows
"""

import pandas as pd

# ── CONFIG ────────────────────────────────────────────────────────────────────
PRODUCTS_FILE    = "final_combinedlast.csv"
INGREDIENTS_FILE = "ingredients_updatedlast.csv"
BRIDGE_FILE      = "bridge_finallast.csv"

OUTPUT_REPORT          = "audit_report.txt"
OUTPUT_BAD_PROD_REFS   = "audit_invalid_product_refs.csv"
OUTPUT_BAD_ING_REFS    = "audit_invalid_ingredient_refs.csv"
OUTPUT_NAME_MISMATCHES = "audit_name_mismatches.csv"
OUTPUT_NO_BRIDGE       = "audit_products_no_bridge.csv"

EXPECTED_ID_LENGTH  = 16
KNOWN_SOURCES       = {'skincarisma', 'datasheet', 'dermstore',
                       'cosmetics', 'skincare_products', 'extracted_from_products'}


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():

    lines  = []
    errors = []
    warns  = []

    def log(msg=''):
        print(msg)
        lines.append(str(msg))

    def err(msg):
        log(f"  ❌ {msg}")
        errors.append(msg)

    def ok(msg):
        log(f"  ✅ {msg}")

    def info(msg):
        log(f"  ℹ️  {msg}")

    def warn(msg):
        log(f"  ⚠️  {msg}")
        warns.append(msg)

    # ── Load ──────────────────────────────────────────────────────────────────
    log("Loading tables ...")
    prod   = pd.read_csv(PRODUCTS_FILE,    low_memory=False)
    ing    = pd.read_csv(INGREDIENTS_FILE, low_memory=False)
    bridge = pd.read_csv(BRIDGE_FILE,      low_memory=False)

    log(f"  Products     : {len(prod):,} rows  |  {len(prod.columns)} columns")
    log(f"  Ingredients  : {len(ing):,} rows  |  {len(ing.columns)} columns")
    log(f"  Bridge       : {len(bridge):,} rows  |  {len(bridge.columns)} columns")

    # Pre-build lookup sets
    prod_ids = set(prod['product_id'].dropna().astype(str))
    ing_ids  = set(ing['ingredient_id'].dropna().astype(str))
    bridge_prod_ids = set(bridge['product_id'].dropna().astype(str))
    bridge_ing_ids  = set(bridge['ingredient_id'].dropna().astype(str))

    # ── CHECK 1: Required columns ──────────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 1 — REQUIRED COLUMNS")
    log("=" * 70)

    required = {
        "Products"   : (prod,   {"product_id", "product_name", "source"}),
        "Ingredients": (ing,    {"ingredient_id", "ingredient_name", "source"}),
        "Bridge"     : (bridge, {"product_id", "ingredient_id",
                                  "product_name", "ingredient_name", "source"}),
    }
    for name, (df, cols) in required.items():
        missing = cols - set(df.columns)
        if missing:
            err(f"{name}: missing columns {missing}")
        else:
            ok(f"{name}: all required columns present")

    # ── CHECK 2: Null IDs ──────────────────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 2 — NULL IDS")
    log("=" * 70)

    null_checks = {
        "products.product_id"      : prod["product_id"].isna().sum(),
        "ingredients.ingredient_id": ing["ingredient_id"].isna().sum(),
        "bridge.product_id"        : bridge["product_id"].isna().sum(),
        "bridge.ingredient_id"     : bridge["ingredient_id"].isna().sum(),
    }
    for label, cnt in null_checks.items():
        if cnt == 0:
            ok(f"{label:<35}: {cnt:,} nulls")
        else:
            err(f"{label:<35}: {cnt:,} nulls found")

    # ── CHECK 3: PK uniqueness ─────────────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 3 — PRIMARY KEY UNIQUENESS")
    log("=" * 70)

    prod_dup = prod["product_id"].duplicated().sum()
    ing_dup  = ing["ingredient_id"].duplicated().sum()

    if prod_dup == 0:
        ok(f"products.product_id: no duplicates")
    else:
        err(f"products.product_id: {prod_dup:,} duplicate values")

    if ing_dup == 0:
        ok(f"ingredients.ingredient_id: no duplicates")
    else:
        err(f"ingredients.ingredient_id: {ing_dup:,} duplicate values")

    # ── CHECK 4: Bridge unique pairs ───────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 4 — BRIDGE UNIQUE PAIRS (product_id + ingredient_id)")
    log("=" * 70)

    bridge_dup = bridge.duplicated(subset=["product_id", "ingredient_id"]).sum()
    if bridge_dup == 0:
        ok("bridge unique on (product_id + ingredient_id)")
    else:
        err(f"bridge: {bridge_dup:,} duplicate (product_id, ingredient_id) pairs")

    # ── CHECK 5: FK bridge.product_id → products ───────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 5 — FK bridge.product_id → products")
    log("=" * 70)

    invalid_prod = bridge[~bridge["product_id"].astype(str).isin(prod_ids)]
    if len(invalid_prod) == 0:
        ok(f"all bridge product_id values exist in products table")
    else:
        err(f"{len(invalid_prod):,} bridge rows reference a product_id not in products table")
        log(f"    Sample invalid product_ids: {invalid_prod['product_id'].unique()[:5].tolist()}")

    # ── CHECK 6: FK bridge.ingredient_id → ingredients ────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 6 — FK bridge.ingredient_id → ingredients")
    log("=" * 70)

    invalid_ing = bridge[~bridge["ingredient_id"].astype(str).isin(ing_ids)]
    if len(invalid_ing) == 0:
        ok("all bridge ingredient_id values exist in ingredients table")
    else:
        err(f"{len(invalid_ing):,} bridge rows reference an ingredient_id not in ingredients table")
        log(f"    Sample invalid ingredient_ids: {invalid_ing['ingredient_id'].unique()[:5].tolist()}")

    # ── CHECK 7: ID format (all 16-char) ──────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 7 — ID FORMAT (all must be 16-char)")
    log("=" * 70)

    id_series = {
        "products.product_id"      : prod["product_id"].dropna().astype(str),
        "ingredients.ingredient_id": ing["ingredient_id"].dropna().astype(str),
        "bridge.product_id"        : bridge["product_id"].dropna().astype(str),
        "bridge.ingredient_id"     : bridge["ingredient_id"].dropna().astype(str),
    }
    for label, series in id_series.items():
        lengths = sorted(series.apply(len).unique())
        all_ok  = all(l == EXPECTED_ID_LENGTH for l in lengths)
        if all_ok:
            ok(f"{label:<35}: {lengths} ✓")
        else:
            err(f"{label:<35}: mixed lengths {lengths} — expected all {EXPECTED_ID_LENGTH}")

    # ── CHECK 8: ID type consistency ───────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 8 — ID TYPE CONSISTENCY")
    log("=" * 70)

    for label, series in [
        ("products.product_id",       prod["product_id"]),
        ("ingredients.ingredient_id", ing["ingredient_id"]),
        ("bridge.product_id",         bridge["product_id"]),
        ("bridge.ingredient_id",      bridge["ingredient_id"]),
    ]:
        dtype = series.dtype
        if dtype == object:
            ok(f"{label:<35}: dtype=string (object) ✓")
        else:
            warn(f"{label:<35}: dtype={dtype} — recommend storing as string")

    # ── CHECK 9: Product coverage ──────────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 9 — PRODUCT COVERAGE")
    log("=" * 70)

    unref_prods = prod[~prod["product_id"].astype(str).isin(bridge_prod_ids)]
    covered_pct = 100 * (len(prod) - len(unref_prods)) / len(prod)

    log(f"  Total products              : {len(prod):,}")
    log(f"  With bridge rows            : {len(prod) - len(unref_prods):,}  ({covered_pct:.1f}%)")
    log(f"  Without bridge rows         : {len(unref_prods):,}")

    has_ing_list   = unref_prods[unref_prods['ingredients_list'].notna()] if 'ingredients_list' in unref_prods.columns else pd.DataFrame()
    no_ing_list    = unref_prods[unref_prods['ingredients_list'].isna()]  if 'ingredients_list' in unref_prods.columns else unref_prods

    log(f"    → No ingredients_list     : {len(no_ing_list):,}  (no source data — expected)")
    log(f"    → Has list but no match   : {len(has_ing_list):,}  (ingredient names not in table)")

    log(f"\n  Coverage by source:")
    for source, grp in prod.groupby('source'):
        covered = grp[grp['product_id'].astype(str).isin(bridge_prod_ids)]
        pct     = 100 * len(covered) / len(grp)
        missing = len(grp) - len(covered)
        icon    = '✅' if pct == 100 else '⚠️ ' if pct < 95 else 'ℹ️ '
        log(f"    {icon} {source:<25}: {len(covered):>6,}/{len(grp):>6,} ({pct:.1f}%)  | {missing:>5,} missing")

    # ── CHECK 10: Ingredient coverage ──────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 10 — INGREDIENT COVERAGE")
    log("=" * 70)

    unref_ings  = ing[~ing["ingredient_id"].astype(str).isin(bridge_ing_ids)]
    ref_ings    = len(ing) - len(unref_ings)
    ing_ref_pct = 100 * ref_ings / len(ing)

    info(f"Ingredients referenced in bridge   : {ref_ings:,}/{len(ing):,} ({ing_ref_pct:.1f}%)")
    info(f"Ingredients NOT in any bridge row  : {len(unref_ings):,}  (not a violation)")

    if 'source' in ing.columns:
        log(f"\n  Unreferenced by source:")
        for source, grp in ing.groupby('source'):
            unref = grp[~grp['ingredient_id'].astype(str).isin(bridge_ing_ids)]
            log(f"    {source:<30}: {len(unref):,} unreferenced")

    # ── CHECK 11: ingredient_name consistency ──────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 11 — INGREDIENT NAME CONSISTENCY (bridge vs ingredients table)")
    log("=" * 70)

    if 'ingredient_name' in bridge.columns:
        id_to_ing_name = dict(zip(ing['ingredient_id'].astype(str), ing['ingredient_name']))

        bridge['_table_ing_name'] = bridge['ingredient_id'].astype(str).map(id_to_ing_name)
        name_mismatch = bridge[
            bridge['_table_ing_name'].notna() &
            (bridge['ingredient_name'].astype(str).str.strip().str.lower() !=
             bridge['_table_ing_name'].astype(str).str.strip().str.lower())
        ]
        bridge = bridge.drop(columns=['_table_ing_name'])

        if len(name_mismatch) == 0:
            ok("all bridge ingredient_name values match ingredients table")
        else:
            warn(f"{len(name_mismatch):,} bridge rows have ingredient_name that differs from ingredients table")
    else:
        info("ingredient_name column not in bridge — skipping")

    # ── CHECK 12: product_name consistency ─────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 12 — PRODUCT NAME CONSISTENCY (bridge vs products table)")
    log("=" * 70)

    if 'product_name' in bridge.columns:
        id_to_prod_name = dict(zip(prod['product_id'].astype(str), prod['product_name']))

        bridge['_table_prod_name'] = bridge['product_id'].astype(str).map(id_to_prod_name)
        prod_name_mismatch = bridge[
            bridge['_table_prod_name'].notna() &
            (bridge['product_name'].astype(str).str.strip().str.lower() !=
             bridge['_table_prod_name'].astype(str).str.strip().str.lower())
        ]
        bridge = bridge.drop(columns=['_table_prod_name'])

        if len(prod_name_mismatch) == 0:
            ok("all bridge product_name values match products table")
        else:
            warn(f"{len(prod_name_mismatch):,} bridge rows have product_name that differs from products table")
    else:
        info("product_name column not in bridge — skipping")

    # ── CHECK 13: Source column ────────────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 13 — SOURCE COLUMN")
    log("=" * 70)

    for label, df in [("Products", prod), ("Ingredients", ing), ("Bridge", bridge)]:
        if 'source' not in df.columns:
            warn(f"{label}: no source column")
            continue
        null_src    = df['source'].isna().sum()
        unknown_src = set(df['source'].dropna().unique()) - KNOWN_SOURCES
        if null_src == 0:
            ok(f"{label}: no null source values")
        else:
            warn(f"{label}: {null_src:,} null source values")
        if not unknown_src:
            ok(f"{label}: all source values are known")
        else:
            warn(f"{label}: unknown source values found: {unknown_src}")

    # ── CHECK 14: Bridge composition ───────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 14 — BRIDGE COMPOSITION BY SOURCE")
    log("=" * 70)

    for source, count in bridge['source'].value_counts().items():
        pct = 100 * count / len(bridge)
        log(f"  {source:<30}: {count:,}  ({pct:.1f}%)")

    # ── CHECK 15: Summary counts ───────────────────────────────────────────────
    log("\n" + "=" * 70)
    log("CHECK 15 — SUMMARY COUNTS")
    log("=" * 70)

    log(f"  Products total         : {len(prod):,}")
    log(f"  Ingredients total      : {len(ing):,}")
    log(f"  Bridge total           : {len(bridge):,}")
    log(f"  Unique products in bridge    : {bridge['product_id'].nunique():,}")
    log(f"  Unique ingredients in bridge : {bridge['ingredient_id'].nunique():,}")
    log(f"  Avg ingredients per product  : {len(bridge) / max(bridge['product_id'].nunique(), 1):.1f}")

    # ── Final verdict ──────────────────────────────────────────────────────────
    log("\n" + "=" * 70)
    if len(errors) == 0 and len(warns) == 0:
        log("✅ OVERALL: FULLY PASSED — no errors, no warnings")
        log("   Safe for Spark / PostgreSQL / Snowflake / Power BI")
    elif len(errors) == 0:
        log(f"⚠️  OVERALL: PASSED WITH WARNINGS ({len(warns)} warning(s))")
        log("   Safe to use but review warnings above")
        for w in warns:
            log(f"   ⚠️  {w}")
    else:
        log(f"❌ OVERALL: FAILED — {len(errors)} error(s), {len(warns)} warning(s)")
        log("   Fix errors before loading to database:")
        for e in errors:
            log(f"   ❌ {e}")
        if warns:
            log("   Also review warnings:")
            for w in warns:
                log(f"   ⚠️  {w}")
    log("=" * 70)

    # ── Save outputs ──────────────────────────────────────────────────────────
    with open(OUTPUT_REPORT, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"\nSaved: {OUTPUT_REPORT}")

    if len(invalid_prod) > 0:
        invalid_prod.to_csv(OUTPUT_BAD_PROD_REFS, index=False)
        print(f"Saved: {OUTPUT_BAD_PROD_REFS}  ({len(invalid_prod):,} rows)")

    if len(invalid_ing) > 0:
        invalid_ing.to_csv(OUTPUT_BAD_ING_REFS, index=False)
        print(f"Saved: {OUTPUT_BAD_ING_REFS}  ({len(invalid_ing):,} rows)")

    if len(unref_prods) > 0:
        cols = ['product_id', 'product_name', 'source']
        if 'ingredients_list' in unref_prods.columns:
            cols.append('ingredients_list')
        unref_prods[cols].to_csv(OUTPUT_NO_BRIDGE, index=False)
        print(f"Saved: {OUTPUT_NO_BRIDGE}  ({len(unref_prods):,} rows)")


if __name__ == "__main__":
    main()

Loading tables ...
  Products     : 24,109 rows  |  47 columns
  Ingredients  : 31,922 rows  |  11 columns
  Bridge       : 653,021 rows  |  9 columns

CHECK 1 — REQUIRED COLUMNS
  ✅ Products: all required columns present
  ✅ Ingredients: all required columns present
  ✅ Bridge: all required columns present

CHECK 2 — NULL IDS
  ✅ products.product_id                : 0 nulls
  ✅ ingredients.ingredient_id          : 0 nulls
  ✅ bridge.product_id                  : 0 nulls
  ✅ bridge.ingredient_id               : 0 nulls

CHECK 3 — PRIMARY KEY UNIQUENESS
  ✅ products.product_id: no duplicates
  ✅ ingredients.ingredient_id: no duplicates

CHECK 4 — BRIDGE UNIQUE PAIRS (product_id + ingredient_id)
  ✅ bridge unique on (product_id + ingredient_id)

CHECK 5 — FK bridge.product_id → products
  ✅ all bridge product_id values exist in products table

CHECK 6 — FK bridge.ingredient_id → ingredients
  ✅ all bridge ingredient_id values exist in ingredients table

CHECK 7 — ID FORMAT (all must be 1

In [44]:
# sorting all 3 files 
import pandas as pd

# =========================================================
# LOAD FILES
# =========================================================
prod = pd.read_csv("final_combinedlast.csv", low_memory=False)
ing = pd.read_csv("ingredients_updatedlast.csv", low_memory=False)
bridge = pd.read_csv("bridge_finallast.csv", low_memory=False)

print("Loaded:")
print("Products   :", len(prod))
print("Ingredients:", len(ing))
print("Bridge     :", len(bridge))


# =========================================================
# SORT PRODUCTS BY NAME
# =========================================================
prod_sorted = prod.sort_values(
    by=["product_name"],
    key=lambda col: col.str.lower()
).reset_index(drop=True)

print("✅ Products sorted by product_name")


# =========================================================
# SORT INGREDIENTS BY NAME
# =========================================================
ing_sorted = ing.sort_values(
    by=["ingredient_name"],
    key=lambda col: col.str.lower()
).reset_index(drop=True)

print("✅ Ingredients sorted by ingredient_name")


# =========================================================
# SORT BRIDGE BY PRODUCT NAME + INGREDIENT NAME
# =========================================================
bridge_sorted = bridge.sort_values(
    by=["product_name", "ingredient_name"],
    key=lambda col: col.str.lower()
).reset_index(drop=True)

print("✅ Bridge sorted by product_name + ingredient_name")


# =========================================================
# SAVE FILES
# =========================================================
prod_sorted.to_csv("ProductTable.csv", index=False)
ing_sorted.to_csv("IngredientsTable.csv", index=False)
bridge_sorted.to_csv("BridgeTable.csv", index=False)


# =========================================================
# SUMMARY
# =========================================================
print("\n" + "=" * 50)
print("SORT COMPLETE (NAME-BASED)")
print("=" * 50)

print("Products   :", len(prod_sorted))
print("Ingredients:", len(ing_sorted))
print("Bridge     :", len(bridge_sorted))

print("\nSaved:")
print("- ProductTable.csv")
print("- IngredientsTable.csv")
print("- BridgeTable.csv")

print("=" * 50)

Loaded:
Products   : 24109
Ingredients: 31922
Bridge     : 653021
✅ Products sorted by product_name
✅ Ingredients sorted by ingredient_name
✅ Bridge sorted by product_name + ingredient_name

SORT COMPLETE (NAME-BASED)
Products   : 24109
Ingredients: 31922
Bridge     : 653021

Saved:
- ProductTable.csv
- IngredientsTable.csv
- BridgeTable.csv
